# 02, Data Preparation (casos)

**O projeto.** Um agente de IA (inteligência artificial) que apoia o analista de
crédito na análise manual de pedidos de financiamento imobiliário do FHA. Para cada
pedido, o agente recomenda aprovar, aprovar com fator compensatório, negar ou
suspender pedindo documentos, e cita a seção do HUD Handbook 4000.1 que sustenta a
recomendação. O notebook 01 mostrou que as bases públicas permitem comparar essa
recomendação com a decisão real dos bancos.

**O que este notebook faz.** Prepara os casos em que o agente vai ser medido, nesta
ordem:

1. Sorteia o grupo de ajuste e o grupo da nota final, antes de qualquer remoção
   (seção 1).
2. Decide quais pedidos do HMDA entram na comparação com o banco (seções 2 a 4). Cada
   remoção aparece com a evidência antes e a contagem depois.
3. Marca, sem tirar da base, os pedidos que precisam de resultado separado (seção 5).
4. Recorta os casos da NSMO para o segundo teste (seção 6).
5. Calcula a resposta esperada pela norma em código, monta o texto que o agente lê e
   grava as saídas (seções 7 a 10).

**A pergunta que a avaliação vai responder.** Nos pedidos cuja negativa tem um motivo
que aparece nos números, o agente chega à decisão do banco? As remoções das seções 2 a
4 existem para que a comparação responda a essa pergunta. O Handbook dividido em
trechos e o índice da norma ficam para o notebook 03.

**Siglas usadas ao longo do notebook.**

| Sigla | Significado | O que é no projeto |
|---|---|---|
| FHA | Federal Housing Administration | órgão do governo americano que garante financiamentos imobiliários de entrada baixa; o projeto trata só desses pedidos |
| HUD | Department of Housing and Urban Development | ministério da habitação dos EUA, dono do FHA e autor da norma (Handbook 4000.1) |
| HMDA | Home Mortgage Disclosure Act | lei que obriga os bancos a publicar seus pedidos de financiamento imobiliário; dá nome à base do teste principal |
| CFPB | Consumer Financial Protection Bureau | agência de defesa do consumidor financeiro que publica o HMDA |
| NSMO | National Survey of Mortgage Originations | pesquisa com quem pegou financiamento; traz score e reserva, mas só de aprovados |
| FHFA | Federal Housing Finance Agency | agência que publica a NSMO junto com o CFPB |
| NMDB | National Mortgage Database | base de financiamentos da FHFA e do CFPB de onde sai a amostra da NSMO |
| DTI | Debt to Income | soma das dívidas mensais (com a nova prestação) dividida pela renda mensal |
| PTI | Payment to Income | só a prestação do imóvel dividida pela renda mensal |
| LTV | Loan to Value | valor financiado dividido pelo valor do imóvel |
| CLTV | Combined Loan to Value | como o LTV, somando todas as dívidas garantidas pelo imóvel |
| AUS | Automated Underwriting System | sistema automático de aprovação do banco |
| LEI | Legal Entity Identifier | código que identifica o banco; usado só para agrupar pedidos por banco, nunca entra no perfil do agente |

**Termos da norma usados ao longo do notebook.**

- *Finalidades.* Três aparecem no notebook inteiro, a compra, o refinanciamento sem
  saque, que troca o financiamento por outro sem tirar dinheiro, e o refinanciamento
  com saque, em que o novo financiamento é maior que o antigo e o cliente leva a
  diferença em dinheiro.
- *Fator compensatório.* Um ponto forte do cliente que a norma aceita para
  compensar um DTI mais alto, como uma reserva de pelo menos três prestações.
- *Tabela manual.* Os limites de PTI e DTI que o Handbook impõe na análise manual,
  conforme o score e os fatores compensatórios. Com score a partir de 580, sem
  fator, o limite é PTI de 31% e DTI de 43%; com 1 fator, 37% e 47%; com 2 fatores,
  40% e 50%. Abaixo de 580, só vale o limite sem fator.

**Fontes.** HMDA público de 2025 (CFPB), EUA inteiro, só pedidos FHA. NSMO Public Use
File, versão 70. As duas bases são lidas de recortes em parquet dos arquivos
originais, com todas as linhas e só as colunas usadas; a origem, as datas de download
e os SHA-256 (Secure Hash Algorithm, a impressão digital de cada arquivo) estão em
`data/README.md`. Aviso exigido pela FHFA ao citar a NSMO:

> The opinions and analyses contained herein are solely of the users/authors of any
> data analyses or papers, and the FHFA cannot and does not attest to nor vouch for
> the quality, accuracy, or timeliness of the data, or analyses derived from these
> data after the data has been retrieved from FHFA.gov.

**Norma.** O gabarito segue o HUD Handbook 4000.1, Update 18, publicada em 12 de
agosto de 2026. Os casos são mais antigos. Os da NSMO são empréstimos de 2019 a 2024,
e os do HMDA foram decididos em 2025, a maior parte antes da Update 17 (26 de novembro
de 2025). A
seção 6.1 procura as regras do gabarito em nove versões do Handbook, de 2016 a 2026,
e todas aparecem iguais, menos o teto do refinanciamento com saque, que era 85% até
2019. As versões entre novembro de 2020 e a Update 17 não foram baixadas, mas as duas
pontas trazem as mesmas regras.

**Sumário**

- [0. Setup](#secao-0). Bibliotecas, funções do projeto e carga das duas bases.
- [1. Sorteio](#secao-1). O HMDA é dividido em 30% para ajuste e 70% para a nota
  final, antes de qualquer remoção.
- [2. Desfechos](#secao-2). Quais desfechos do pedido trazem uma decisão de crédito
  para comparar.
- [3. Negativas](#secao-3). Quais negativas têm o motivo visível nos campos
  publicados (DTI e garantia).
- [4. Qualidade do dado](#secao-4). Renda, campos "Exempt", DTI vazio, ocupação e
  finalidades sem teto; o que fica sem regra especial; o funil completo.
- [5. Marcações](#secao-5). Colunas que separam os resultados na Evaluation, sem tirar
  pedidos da base.
- [6. NSMO](#secao-6). Casos de 44% a 50% com score e reserva: finalidade, menor
  score e grupos.
- [7. Resposta esperada](#secao-7). O que a norma em código responde em cada caso,
  os testes e a comparação com as referências.
- [8. O que o agente vê](#secao-8). O perfil do caso em texto.
- [9. Contagens e conjuntos fixos](#secao-9). Casos por célula no grupo de ajuste e
  os conjuntos tirados dele.
- [10. Saídas](#secao-10). Arquivos gravados em `data/processed/`.
- [11. Limitações](#secao-11).
- [12. Síntese](#secao-12).

<a id="secao-0"></a>

## 0. Setup

In [1]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.case_profile import case_profile, hmda_case, nsmo_case, nsmo_purpose_code
from src.handbook import HANDBOOK, has_sentence
from src.hmda import (
    ACTION_TAKEN,
    DENIAL_REASON,
    HMDA_PARQUET,
    LIEN,
    LOAN_PURPOSE,
    OCCUPANCY,
    dti_band,
    load_hmda,
)
from src.nsmo import (
    LOAN_TYPE_FHA,
    NSMO_PARQUET,
    NSMO_TERMS,
    X33,
    load_nsmo,
)
from src.rules import (
    APPROVE,
    CLTV_CAP,
    DENY,
    UFMIP,
    cap_with_ufmip,
    cltv_situation,
    expected_hmda,
    expected_nsmo,
    in_ufmip_band,
    manual_requirement,
    norm_reading,
    split_answers,
)

SEED = 42
SHARE_TUNING = 0.30  # 30% para o grupo de ajuste, 70% para a nota final
PROCESSED = ROOT / 'data' / 'processed'
FAIXAS = ['até 43', '44 a 49', '50 a 60', 'acima de 60']
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)


def br(n):
    """Inteiro com ponto de milhar."""
    return f'{int(n):,}'.replace(',', '.')


def pct(parte, total):
    """Percentual com uma casa e vírgula decimal."""
    return f'{100 * parte / total:.1f}%'.replace('.', ',')

In [2]:
hmda = load_hmda()
COLUNAS_HMDA = list(hmda.columns)
print(HMDA_PARQUET.name, hmda.shape)

assert NSMO_TERMS.exists(), 'cópia dos Terms of Service da FHFA ausente'
nsmo = load_nsmo()
COLUNAS_NSMO = list(nsmo.columns)
print(NSMO_PARQUET.name, nsmo.shape)

hmda_2025_fha.parquet (2062934, 19)
nsmo_v70_1324_puf.parquet (62359, 18)


<a id="secao-1"></a>

## 1. Sorteio do grupo de ajuste e do grupo da nota final

**A pergunta.** Como separar os casos usados para ajustar o agente dos casos que dão
a nota final?

O agente não aprende com os dados como um modelo treinado. O risco está em quem
escreve o prompt, que, ao ver os erros, ajusta o texto até o agente acertar aqueles
casos.
Se a nota final viesse dos mesmos casos, mediria o quanto o prompt se adaptou a eles.
Por isso a base é dividida em dois grupos:

- **grupo de ajuste (30%)**, onde o prompt é testado e corrigido quantas vezes for
  preciso;
- **grupo da nota final (70%)**, que roda uma vez só, na Evaluation, com o prompt
  congelado.

A divisão fica em 30% e 70% porque o ajuste só precisa de casos suficientes para
revisar o prompt, e a nota final ganha com mais casos em cada combinação de faixa de
DTI e finalidade. O agente não roda nos milhões de pedidos. Na Evaluation, uma amostra
de tamanho fixo sai de dentro de cada grupo.

O sorteio acontece aqui, na base inteira e antes de qualquer remoção. As remoções das
seções seguintes valem igual para os dois grupos. As seções 2 e 3 aplicam decisões
do gate, tomadas com o notebook 01, antes do sorteio. A partir da seção 4, as regras
novas olham a decisão do banco só no grupo de ajuste, e a decisão do banco no grupo da
nota final só é contada na Evaluation. Dentro de cada finalidade, 30% dos pedidos vão
para o ajuste, com
semente fixa (42) para o sorteio sair igual em toda execução.

In [3]:
rng = np.random.default_rng(SEED)
hmda['grupo'] = 'nota final'
for finalidade in sorted(hmda['loan_purpose'].unique()):
    indices = hmda.index[hmda['loan_purpose'] == finalidade].to_numpy()
    embaralhados = rng.permutation(indices)
    n_ajuste = round(len(indices) * SHARE_TUNING)
    hmda.loc[embaralhados[:n_ajuste], 'grupo'] = 'ajuste'

hmda['finalidade'] = hmda['loan_purpose'].map(LOAN_PURPOSE)
print('pedidos por grupo:', hmda['grupo'].value_counts().to_dict())
proporcao = pd.crosstab(hmda['finalidade'], hmda['grupo'], normalize='index')
(proporcao * 100).round(1)

pedidos por grupo: {'nota final': 1444053, 'ajuste': 618881}


grupo,ajuste,nota final
finalidade,,
compra,30.0,70.0
não se aplica,30.0,70.0
outro,30.0,70.0
refinanciamento com saque,30.0,70.0
refinanciamento sem saque,30.0,70.0
reforma,30.0,70.0


In [4]:
hmda['desfecho'] = hmda['action_taken'].map(ACTION_TAKEN)
por_desfecho = pd.crosstab(hmda['desfecho'], hmda['grupo'], normalize='index')
(por_desfecho * 100).round(1)

grupo,ajuste,nota final
desfecho,,
"aprovado, não aceito",30.4,69.6
comprado de terceiro,29.9,70.1
fechado por incompletude,29.8,70.2
negado,30.0,70.0
originado,30.0,70.0
"pré aprovação aprovada, não aceita",30.0,70.0
pré aprovação negada,30.4,69.6
retirado pelo cliente,29.9,70.1


**Onde olhar:** a coluna "ajuste" das duas tabelas.

**O que se vê:** 618.881 pedidos no ajuste e 1.444.053 na nota final. Toda finalidade
fica com 30,0% no ajuste, porque o sorteio foi feito dentro de cada uma. Os
desfechos, que não entraram no sorteio, ficam entre 29,8% e 30,4%.

**O que significa:** os dois grupos têm a mesma composição. Uma diferença de acerto
entre eles na Evaluation não vai poder ser explicada pela mistura de finalidades ou de
desfechos.

<a id="secao-2"></a>

## 2. HMDA: quais desfechos entram

**A pergunta.** Quais pedidos trazem uma decisão do banco que serve de gabarito?

A comparação precisa de pedidos em que o banco olhou os números e decidiu, aprovando
ou negando. A coluna `action_taken` diz como cada pedido terminou, e esta seção passa
pelos oito desfechos.

A célula abaixo prepara o que o resto do notebook usa:

- colunas de apoio: decisão (aprovado ou negado), DTI e CLTV em número, faixa de DTI;
- **motivo visível**: DTI ou garantia declarado entre os quatro motivos da negativa,
  com o campo correspondente publicado (DTI com o DTI, garantia com o CLTV);
- a função `remover`, usada em toda remoção. Ela grava o motivo da saída na coluna
  `sai_por` e imprime quantos pedidos saem, por desfecho. Nenhuma linha é apagada, e
  essa coluna monta o funil da seção 4.7.

In [5]:
hmda['decisao'] = hmda['action_taken'].map(
    {'1': 'aprovado', '2': 'aprovado', '3': 'negado'}
)
hmda['tem_dti'] = hmda['debt_to_income_ratio'].notna() & (
    hmda['debt_to_income_ratio'] != 'Exempt'
)
hmda['tem_cltv'] = hmda['loan_to_value_ratio'].notna() & (
    hmda['loan_to_value_ratio'] != 'Exempt'
)
hmda['renda'] = pd.to_numeric(hmda['income'], errors='coerce')
hmda['valor'] = pd.to_numeric(hmda['loan_amount'], errors='coerce')
hmda['cltv'] = pd.to_numeric(hmda['loan_to_value_ratio'], errors='coerce')
hmda['faixa_dti'] = dti_band(hmda['debt_to_income_ratio'])

MOTIVOS = ['denial_reason-1', 'denial_reason-2', 'denial_reason-3', 'denial_reason-4']
motivos = hmda[MOTIVOS]
hmda['motivo_dti_visivel'] = (motivos == '1').any(axis=1) & hmda['tem_dti']
hmda['motivo_gar_visivel'] = (motivos == '4').any(axis=1) & hmda['tem_cltv']
hmda['motivo_visivel'] = hmda['motivo_dti_visivel'] | hmda['motivo_gar_visivel']

# Cada pedido removido guarda aqui o motivo da remoção; vazio = continua na base.
hmda['sai_por'] = ''


def na_base():
    """Pedidos que ainda não saíram por nenhuma decisão."""
    return hmda['sai_por'] == ''


def no_ajuste():
    """Pedidos da base que estão no grupo de ajuste."""
    return na_base() & (hmda['grupo'] == 'ajuste')


def remover(mascara, motivo):
    """Marca os pedidos da base que atendem à máscara e imprime a contagem."""
    alvo = na_base() & mascara
    hmda.loc[alvo, 'sai_por'] = motivo
    print(f'{motivo}: saem {br(alvo.sum())} pedidos')
    saem = hmda.loc[alvo, 'desfecho'].value_counts()
    for desfecho, n in saem.items():
        print(f'  {desfecho}: {br(n)}')
    print(f'ficam na base: {br(na_base().sum())}')

### 2.1 Pedidos sem decisão de crédito

**Pergunta:** quantos pedidos caem em cada desfecho, e quais não têm decisão de
crédito do banco?

In [6]:
contagem = hmda['desfecho'].value_counts()
tabela = pd.DataFrame({
    'pedidos': contagem,
    '% da base': (100 * contagem / len(hmda)).round(1),
})
tabela

,pedidos,% da base
desfecho,,
originado,906942,44.0
comprado de terceiro,427636,20.7
retirado pelo cliente,282798,13.7
negado,265894,12.9
fechado por incompletude,102018,4.9
"aprovado, não aceito",44461,2.2
"pré aprovação aprovada, não aceita",20765,1.0
pré aprovação negada,12420,0.6


**Onde olhar:** as linhas "comprado de terceiro", "retirado pelo cliente" e as duas de
pré aprovação.

**O que se vê:** 427.636 comprados de terceiro (20,7%), 282.798 retirados pelo
cliente (13,7%), 20.765 pré aprovações aprovadas e não aceitas e 12.420 pré aprovações
negadas.

**O que significa:** nenhum desses quatro desfechos traz uma decisão de crédito do
banco sobre o pedido.

- *Comprado de terceiro*: quem decidiu foi outro banco; quem comprou só registra a
  compra.
- *Retirado pelo cliente*: o cliente desistiu antes da decisão.
- *Pré aprovação*: consulta feita antes de o cliente escolher o imóvel, sem imóvel
  definido para medir a garantia.

Os quatro saem juntos, 743.619 pedidos.

In [7]:
sem_decisao = hmda['action_taken'].isin(['4', '6', '7', '8'])
remover(sem_decisao, 'sem decisão de crédito')

sem decisão de crédito: saem 743.619 pedidos
  comprado de terceiro: 427.636
  retirado pelo cliente: 282.798
  pré aprovação aprovada, não aceita: 20.765
  pré aprovação negada: 12.420
ficam na base: 1.319.315


### 2.2 Fechados por incompletude

**Pergunta:** os pedidos fechados por incompletude servem de gabarito?

Nesse desfecho, o cliente não entregou um documento pedido e o banco encerrou o
pedido sem decidir. A tabela compara, por desfecho, quantos pedidos chegam sem DTI e
sem CLTV.

In [8]:
linhas = []
for desfecho in ['originado', 'aprovado, não aceito', 'negado',
                 'fechado por incompletude']:
    grupo = hmda[hmda['desfecho'] == desfecho]
    sem_dti = ~grupo['tem_dti']
    sem_cltv = ~grupo['tem_cltv']
    linhas.append({
        'desfecho': desfecho,
        'pedidos': len(grupo),
        'sem DTI (%)': round(100 * sem_dti.mean(), 1),
        'sem CLTV (%)': round(100 * sem_cltv.mean(), 1),
        'sem os dois (%)': round(100 * (sem_dti & sem_cltv).mean(), 1),
    })
pd.DataFrame(linhas).set_index('desfecho')

,pedidos,sem DTI (%),sem CLTV (%),sem os dois (%)
desfecho,,,,
originado,906942,10.9,7.3,6.5
"aprovado, não aceito",44461,37.9,21.2,20.7
negado,265894,14.0,9.5,6.9
fechado por incompletude,102018,100.0,100.0,100.0


**Onde olhar:** a linha "fechado por incompletude", comparada com "originado" e
"negado".

**O que se vê:** os 102.018 fechados por incompletude vêm todos sem DTI e sem CLTV.
Nos originados, o DTI falta em 10,9% e o CLTV em 7,3%; nos negados, em 14,0% e 9,5%.

**O que significa:** sem decisão do banco e sem os dois números que a norma usa,
esses pedidos não têm o que comparar. Saem 102.018. A seção 3.1 volta ao assunto,
porque parte das negativas é o mesmo acontecimento registrado de outro jeito.

In [9]:
remover(hmda['action_taken'] == '5', 'fechado por incompletude')

fechado por incompletude: saem 102.018 pedidos
  fechado por incompletude: 102.018
ficam na base: 1.217.297


### 2.3 Aprovado, não aceito

**Pergunta:** o "aprovado, não aceito" pode contar como aprovação?

Nesse desfecho, o banco aprovou e o cliente não fechou, por exemplo porque achou juros
menores em outro banco. A Regulation C, regra federal que define o que o HMDA publica,
só permite esse código quando a decisão de crédito já foi tomada e faltavam apenas
condições de fechamento, como título e vistoria (comentários 4(a)(8)(i)-3 e -13).
Condição de crédito não cumprida vira "negado", e documento não entregue vira "fechado
por incompletude". Pela regra, a decisão foi aprovar. Falta ver se os números desses
pedidos se parecem com os dos originados.

In [10]:
com_dti = hmda[na_base() & hmda['tem_dti']]
faixas = pd.crosstab(com_dti['faixa_dti'], com_dti['desfecho'], normalize='columns')
(faixas * 100).round(1).reindex(FAIXAS)

desfecho,"aprovado, não aceito",negado,originado
faixa_dti,,,
até 43,43.1,34.4,42.8
44 a 49,26.5,17.2,26.9
50 a 60,28.7,26.0,30.1
acima de 60,1.7,22.4,0.2


In [11]:
aprovados = hmda[na_base() & hmda['action_taken'].isin(['1', '2'])]
sem_dti = pd.crosstab(
    aprovados['finalidade'], aprovados['desfecho'],
    values=~aprovados['tem_dti'], aggfunc='mean',
)
print('DTI vazio por finalidade (%)')
print((sem_dti * 100).round(1))
nao_aceito = aprovados[aprovados['action_taken'] == '2']
acima_60 = (nao_aceito['debt_to_income_ratio'] == '>60%').sum()
print()
print('aprovados, não aceitos:', br(len(nao_aceito)),
      '| com DTI acima de 60%:', br(acima_60))

DTI vazio por finalidade (%)
desfecho                   aprovado, não aceito  originado
finalidade                                                
compra                                      3.6        1.7
não se aplica                               0.0       19.0
outro                                      62.4       54.4
refinanciamento com saque                   6.1        2.0
refinanciamento sem saque                  85.5       68.8
reforma                                    48.0       34.4

aprovados, não aceitos: 44.461 | com DTI acima de 60%: 469


**Onde olhar:** na primeira tabela, as colunas "aprovado, não aceito" e "originado";
na segunda, a linha "refinanciamento sem saque".

**O que se vê:** com o DTI preenchido, as faixas quase coincidem. Até 43% estão 43,1%
dos não aceitos e 42,8% dos originados; de 50% a 60%, 28,7% e 30,1%. O DTI vazio é
mais comum nos não aceitos (37,9% contra 10,9%, na tabela da 2.2), puxado pelo
refinanciamento sem saque (85,5% contra 68,8%). Ali deve estar o Streamline, o
refinanciamento simplificado de quem já tem financiamento FHA, que pode ser feito sem
análise de renda. Há 469 não aceitos com DTI acima de 60% (1,7%, contra 0,2% dos
originados).

**O que significa:** os 44.461 aprovados não aceitos contam como aprovação. O DTI
vazio é tratado na seção 4.3, igual para todos os desfechos. Os 469 com DTI acima de
60% ficam. São aprovações que a análise manual não permitiria, e na comparação contam
como discordância entre a norma e o banco, como qualquer aprovado acima de 60%.

<a id="secao-3"></a>

## 3. HMDA: quais negativas entram

**A pergunta.** Em quais negativas o motivo do banco aparece nos números publicados?

O banco informa até quatro motivos para cada negativa (`denial_reason-1` a `-4`), e a
Regulation C não define ordem de importância entre eles. Só dois motivos têm um campo
correspondente no HMDA. O DTI (as dívidas pesam demais na renda) é conferido pelo DTI
publicado, e a garantia (o imóvel não cobre o empréstimo) é conferida pelo CLTV. Nos
outros motivos, como histórico de crédito ou de emprego, a razão está em documentos
que a base não publica, e nem o agente nem a norma em código chegam a ela pelos
números.

A regra de toda a seção é que a negativa entra se ao menos um dos quatro motivos for
visível (definição na seção 2). A tabela agrupa os negados pelo primeiro motivo
declarado, como no notebook 01, só para descrever a base. Quem fica é decidido pelos
quatro motivos.

In [12]:
def grupo_motivo(codigo):
    """Grupo do 1º motivo declarado, como no notebook 01."""
    if codigo in ('1', '4'):
        return 'visível (DTI ou garantia)'
    if codigo in ('2', '3', '6'):
        return 'invisível (crédito, emprego, informação)'
    if codigo == '7':
        return 'cadastro incompleto'
    return 'outros'


negados = hmda[na_base() & (hmda['decisao'] == 'negado')].copy()
negados['grupo do 1º motivo'] = negados['denial_reason-1'].map(grupo_motivo)
negados['algum motivo visível'] = negados['motivo_visivel'].map(
    {True: 'sim', False: 'não'}
)
pd.crosstab(negados['grupo do 1º motivo'], negados['algum motivo visível'],
            margins=True, margins_name='total')

algum motivo visível,não,sim,total
grupo do 1º motivo,,,
cadastro incompleto,39356,85,39441
"invisível (crédito, emprego, informação)",62800,7922,70722
outros,36078,1175,37253
visível (DTI ou garantia),3617,114861,118478
total,141851,124043,265894


**Onde olhar:** a coluna "sim", linha a linha.

**O que se vê:** dos 265.894 negados, 124.043 têm algum motivo visível. Quando o
primeiro motivo já é DTI ou garantia, quase todos têm (114.861 de 118.478); nos 3.617
restantes, o motivo foi declarado, mas o campo veio vazio. Nos grupos "invisível" e
"outros", só 7.922 e 1.175 trazem DTI ou garantia em outra posição. No cadastro
incompleto, 85.

**O que significa:** cada grupo tem uma subseção a seguir. O cadastro incompleto vem
primeiro, porque sai por um motivo próprio, que não depende dos números.

### 3.1 Cadastro incompleto

**Pergunta:** a negativa por cadastro incompleto serve de gabarito?

"Cadastro incompleto" é o código 7 do motivo da negativa. Quando falta documento, a
Regulation B, regra federal sobre pedidos de crédito, dá ao banco 30 dias para negar
ou mandar um aviso de incompletude (12 CFR 1002.9(c), no CFR, Code of Federal
Regulations). Depois do aviso, a Regulation C
deixa o banco registrar o pedido como "fechado por incompletude" ou como "negado", à
escolha dele (comentário 4(a)(8)(i)-6). A tabela compara esses negados com os outros
negados e com os originados.

In [13]:
negado = na_base() & (hmda['decisao'] == 'negado')
incompleto = negado & (hmda['denial_reason-1'] == '7')
originados = na_base() & (hmda['action_taken'] == '1')
outros_negados = negado & (hmda['denial_reason-1'] != '7')

linhas = []
for nome, mascara in [('cadastro incompleto', incompleto),
                      ('outros negados', outros_negados),
                      ('originados', originados)]:
    grupo = hmda[mascara]
    linhas.append({
        'grupo': nome,
        'pedidos': len(grupo),
        'sem DTI (%)': round(100 * (~grupo['tem_dti']).mean(), 1),
        'sem CLTV (%)': round(100 * (~grupo['tem_cltv']).mean(), 1),
    })
print(pd.DataFrame(linhas).set_index('grupo'))
completos = hmda[incompleto & hmda['tem_dti'] & hmda['tem_cltv']]
print()
print('cadastro incompleto com DTI e CLTV publicados:', br(len(completos)))
print('cadastro incompleto com DTI ou garantia em outro motivo:',
      br((incompleto & hmda['motivo_visivel']).sum()))

                     pedidos  sem DTI (%)  sem CLTV (%)
grupo                                                  
cadastro incompleto    39441         42.3          29.2
outros negados        226453          9.1           6.0
originados            906942         10.9           7.3

cadastro incompleto com DTI e CLTV publicados: 19.418
cadastro incompleto com DTI ou garantia em outro motivo: 85


**Onde olhar:** a linha "cadastro incompleto" e as duas contagens abaixo da tabela.

**O que se vê:** 42,3% desses negados vêm sem DTI e 29,2% sem CLTV, contra 9,1% e
6,0% nos outros negados. Mesmo assim, 19.418 chegam com os dois campos e parecem
pedidos normais, e 85 trazem DTI ou garantia em outro motivo.

**O que significa:** o motivo da negativa é um documento que faltou, e esse documento
não está na base. Pela Regulation C, é o mesmo acontecimento dos fechados por
incompletude da seção 2.2, registrado de outro jeito. Saem os 39.441, inclusive os
19.418 completos e os 85 com motivo visível, porque o critério vem da regra e vale
para todos. Na seção 9, 20 desses pedidos sem DTI ou CLTV, tirados do grupo de ajuste,
ficam separados para uma checagem à parte, que vê se o agente suspende pedindo o dado
ou inventa um número.

In [14]:
remover(incompleto, 'negado por cadastro incompleto')

negado por cadastro incompleto: saem 39.441 pedidos
  negado: 39.441
ficam na base: 1.177.856


### 3.2 1º motivo invisível

**Pergunta:** as negativas com 1º motivo invisível têm algum motivo visível nas
outras posições?

Motivo invisível é histórico de crédito, histórico de emprego ou informação não
verificável. A tabela abre cada um. Mostra quantos têm DTI ou garantia em outra
posição e, nos que não têm, quantos trazem nos números um sinal que a norma
reprovaria (DTI acima de 60%, ou refinanciamento com saque com CLTV acima de 80%, o
teto do FHA para essa finalidade).

In [15]:
def caminhos(codigos):
    """Para os negados com esses 1º motivos, quantos têm motivo visível."""
    negado = na_base() & (hmda['decisao'] == 'negado')
    linhas = []
    for codigo in codigos:
        grupo = hmda[negado & (hmda['denial_reason-1'] == codigo)]
        sem_visivel = grupo[~grupo['motivo_visivel']]
        dti_60 = sem_visivel['debt_to_income_ratio'] == '>60%'
        saque_80 = (sem_visivel['loan_purpose'] == '32') & (sem_visivel['cltv'] > 80)
        linhas.append({
            '1º motivo': DENIAL_REASON[codigo],
            'negados': len(grupo),
            'DTI em outro motivo': int(grupo['motivo_dti_visivel'].sum()),
            'garantia em outro motivo': int(grupo['motivo_gar_visivel'].sum()),
            'algum visível': int(grupo['motivo_visivel'].sum()),
            'nenhum visível': len(sem_visivel),
            'nenhum visível, DTI acima de 60%': int(dti_60.sum()),
            'nenhum visível, saque acima de 80%': int(saque_80.sum()),
        })
    return pd.DataFrame(linhas).set_index('1º motivo')


caminhos(['3', '6', '2'])

,negados,DTI em outro motivo,garantia em outro motivo,algum visível,nenhum visível,"nenhum visível, DTI acima de 60%","nenhum visível, saque acima de 80%"
1º motivo,,,,,,,
histórico de crédito,53720,5712,1121,6635,47085,4086,1291
informação não verificável,12477,506,138,622,11855,807,349
histórico de emprego,4525,589,78,665,3860,467,21


**Onde olhar:** as colunas "algum visível" e "nenhum visível", e as duas últimas.

**O que se vê:** dos 53.720 negados por histórico de crédito, 6.635 têm DTI ou
garantia em outra posição. Nos de informação não verificável são 622 de 12.477, e nos
de emprego, 665 de 4.525. Entre os que não têm motivo visível, alguns mostram DTI
acima de 60% (4.086, 807 e 467, na ordem da tabela) ou saque acima de 80% (1.291, 349
e 21).

**O que significa:** ficam os 7.922 com motivo visível em outra posição, e cada um
segue a regra da sua faixa de DTI, como qualquer pedido. Saem os 62.800 sem motivo
visível, inclusive os que têm DTI acima de 60% ou saque acima de 80%. Nesses, o banco
não declarou DTI nem garantia. Manter só os que o número denuncia seria escolher os
casos em que o agente tem mais chance de acertar.

In [16]:
invisivel = (
    (hmda['decisao'] == 'negado')
    & hmda['denial_reason-1'].isin(['2', '3', '6'])
    & ~hmda['motivo_visivel']
)
remover(invisivel, 'motivo invisível sem motivo visível')

motivo invisível sem motivo visível: saem 62.800 pedidos
  negado: 62.800
ficam na base: 1.115.056


### 3.3 Grupo "outros"

**Pergunta:** e no grupo "outros"?

O grupo junta "outro", recursos insuficientes para o fechamento, "isento" (banco
dispensado de informar o motivo, código 1111) e seguro hipotecário negado. A tabela é
a mesma da 3.2.

In [17]:
caminhos(['9', '5', '1111', '8'])

,negados,DTI em outro motivo,garantia em outro motivo,algum visível,nenhum visível,"nenhum visível, DTI acima de 60%","nenhum visível, saque acima de 80%"
1º motivo,,,,,,,
outro,19785,254,147,367,19418,1213,262
recursos insuficientes,16858,687,120,801,16057,192,213
isento (banco dispensado de informar),526,0,0,0,526,0,0
seguro hipotecário negado,84,5,2,7,77,2,1


**Onde olhar:** as mesmas colunas da 3.2.

**O que se vê:** dos 37.253 negados do grupo, só 1.175 têm DTI ou garantia em outra
posição (367 em "outro", 801 em recursos insuficientes e 7 em seguro negado). Os 526
isentos não trazem motivo nenhum.

**O que significa:** vale a regra da 3.2. Ficam os 1.175 e saem 36.078.

In [18]:
outros = (
    (hmda['decisao'] == 'negado')
    & hmda['denial_reason-1'].isin(['5', '8', '9', '1111'])
    & ~hmda['motivo_visivel']
)
remover(outros, 'grupo outros sem motivo visível')

grupo outros sem motivo visível: saem 36.078 pedidos
  negado: 36.078
ficam na base: 1.078.978


### 3.4 Motivo visível sem o campo publicado

**Pergunta:** sobrou algum negado sem motivo visível?

Depois das três remoções, restam negados que declararam DTI ou garantia, mas sem o
campo correspondente publicado. A célula conta esses pedidos e mostra uma amostra.

In [19]:
negado = na_base() & (hmda['decisao'] == 'negado')
sem_campo = hmda[negado & ~hmda['motivo_visivel']]
sem_campo_motivos = sem_campo[MOTIVOS]
print('negados que restam sem motivo visível com o campo publicado:',
      br(len(sem_campo)))
print('  DTI declarado e DTI não publicado:',
      br(((sem_campo_motivos == '1').any(axis=1) & ~sem_campo['tem_dti']).sum()))
print('  garantia declarada e CLTV não publicado:',
      br(((sem_campo_motivos == '4').any(axis=1) & ~sem_campo['tem_cltv']).sum()))
cols = ['finalidade', 'debt_to_income_ratio', 'loan_to_value_ratio'] + MOTIVOS
sem_campo[cols].sample(min(8, len(sem_campo)), random_state=SEED)

negados que restam sem motivo visível com o campo publicado: 3.617
  DTI declarado e DTI não publicado: 1.691
  garantia declarada e CLTV não publicado: 1.962


,finalidade,debt_to_income_ratio,loan_to_value_ratio,denial_reason-1,denial_reason-2,denial_reason-3,denial_reason-4
757637,refinanciamento sem saque,NaN,NaN,1,NaN,NaN,NaN
1028534,compra,NaN,96.5,1,NaN,NaN,NaN
1438244,refinanciamento com saque,39,NaN,4,NaN,NaN,NaN
1584923,refinanciamento sem saque,NaN,NaN,4,NaN,NaN,NaN
1799189,outro,NaN,NaN,4,9,NaN,NaN
1156004,compra,NaN,89.205,1,NaN,NaN,NaN
1930022,outro,NaN,47.33,1,NaN,NaN,NaN
1794982,refinanciamento sem saque,NaN,NaN,4,9,NaN,NaN


**Onde olhar:** as duas contagens e, na amostra, as colunas `debt_to_income_ratio` e
`loan_to_value_ratio`.

**O que se vê:** são 3.617 negados. Em 1.691, o motivo é DTI e o DTI veio vazio; em
1.962, é garantia e o CLTV veio vazio (um pedido pode estar nos dois). Na amostra, o
pedido 1438244 foi negado por garantia, tem DTI de 39% e nenhum CLTV.

**O que significa:** o motivo foi declarado, mas o número que o mostraria não está na
base. Saem os 3.617. A partir daqui, todo negado que fica tem o motivo declarado e o
campo que o confirma.

In [20]:
sem_campo = (hmda['decisao'] == 'negado') & ~hmda['motivo_visivel']
remover(sem_campo, 'motivo visível sem o campo publicado')

motivo visível sem o campo publicado: saem 3.617 pedidos
  negado: 3.617
ficam na base: 1.075.361


<a id="secao-4"></a>

## 4. HMDA: qualidade do dado

**A pergunta.** Os pedidos que restam trazem números confiáveis, que a norma consegue
avaliar?

As seções 2 e 3 trataram da decisão do banco. Esta seção olha o dado do pedido, como renda,
campos dispensados de publicação, DTI vazio, ocupação do imóvel e finalidade. Cada
regra vale igual para aprovados e negados, sem olhar o acerto que o agente teria.

Nesta seção e na seção 5, toda tabela que mostra a decisão do banco usa só o grupo de
ajuste. A remoção vale para os dois grupos, e a contagem do que sai, impressa depois
de cada remoção, cobre a base inteira.

### 4.1 Renda zero, negativa ou incoerente com o DTI

**Pergunta:** a renda publicada é coerente com o DTI?

Primeiro vem a renda zero ou negativa, com a qual o DTI não tem como existir. A
primeira linha da saída conta esses pedidos.

Depois vem a renda incoerente. A célula estima a prestação de cada pedido (valor
financiado, 6,5% ao ano, 30 anos) e a divide pela renda mensal. O DTI já inclui essa
prestação, então a prestação dividida pela renda nunca pode passar do DTI. Parte dos
DTIs vem em faixas ("30% a menos de 36%", por exemplo), e a conta usa o topo da
faixa, o valor mais folgado. Se a prestação estimada sozinha passa de 2 vezes esse
topo, a renda e o DTI não podem vir do mesmo pedido. A conta só detecta o problema, sem corrigir o dado, e não
vai para o agente. A amostra mostra dez aprovados nessa situação (a renda vem em
milhares de dólares, 10 quer dizer US$ 10 mil).

In [21]:
base = hmda[no_ajuste()]
print('renda zero ou negativa:',
      base.loc[base['renda'] <= 0, 'decisao'].value_counts().to_dict())
print('renda vazia:', base.loc[base['renda'].isna(), 'decisao'].value_counts().to_dict())

# Prestação estimada: valor financiado, 6,5% ao ano, 30 anos, sobre a renda mensal.
# Serve só para detectar incoerência; não vai para o agente.
taxa = 0.065 / 12
fator = taxa / (1 - (1 + taxa) ** -360)
hmda['pti_estimado'] = hmda['valor'] * fator / (hmda['renda'] * 1000 / 12) * 100

# Maior valor possível de cada DTI publicado.
teto_dti = {'<20%': 20, '20%-<30%': 30, '30%-<36%': 36, '50%-60%': 60}
for valor in range(36, 50):
    teto_dti[str(valor)] = valor
hmda['dti_teto'] = hmda['debt_to_income_ratio'].map(teto_dti)
hmda['razao_renda'] = hmda['pti_estimado'] / hmda['dti_teto']

base = hmda[no_ajuste() & (hmda['renda'] > 0)]
incoerente = base[base['razao_renda'] > 2]
cols = ['lei', 'finalidade', 'valor', 'renda', 'debt_to_income_ratio',
        'pti_estimado', 'decisao']
incoerente[incoerente['decisao'] == 'aprovado'][cols].sample(
    10, random_state=SEED
).round(1)

renda zero ou negativa: {'aprovado': 2589, 'negado': 661}
renda vazia: {'aprovado': 24385, 'negado': 230}


,lei,finalidade,valor,renda,debt_to_income_ratio,pti_estimado,decisao
518532,549300HIVO8XPBPNVG69,compra,465000.0,9.0,36,391.9,aprovado
519934,549300HIVO8XPBPNVG69,compra,375000.0,11.0,20%-<30%,258.6,aprovado
1680027,549300DCGBXW5FJMV921,compra,825000.0,18.0,50%-60%,347.6,aprovado
518387,549300HIVO8XPBPNVG69,compra,455000.0,8.0,46,431.4,aprovado
518894,549300HIVO8XPBPNVG69,compra,125000.0,3.0,44,316.0,aprovado
519328,549300HIVO8XPBPNVG69,compra,285000.0,6.0,43,360.3,aprovado
518024,549300HIVO8XPBPNVG69,refinanciamento com saque,315000.0,14.0,<20%,170.7,aprovado
520209,549300HIVO8XPBPNVG69,compra,495000.0,12.0,30%-<36%,312.9,aprovado
518389,549300HIVO8XPBPNVG69,compra,335000.0,11.0,20%-<30%,231.0,aprovado
519622,549300HIVO8XPBPNVG69,compra,245000.0,8.0,20%-<30%,232.3,aprovado


**Onde olhar:** as colunas `renda`, `pti_estimado` e `lei`.

**O que se vê:** rendas de US$ 3 mil a US$ 18 mil por ano para empréstimos de US$ 125
mil a US$ 825 mil, com prestação estimada entre 171% e 431% da renda. Mesmo assim, o
DTI publicado fica entre "abaixo de 20%" e "50% a 60%".

**O que significa:** uma prestação maior que a renda inteira não cabe num DTI abaixo
de 20%. Um dos dois números está errado, e o dado não diz qual. O corte de 2 vezes é
uma escolha do projeto, e por isso a próxima célula refaz a contagem com 1,5 e 3
vezes, para ver se o resultado depende dele.

In [22]:
linhas = []
for corte in [1.5, 2.0, 3.0]:
    acima = base[base['razao_renda'] > corte]
    contagem = acima['decisao'].value_counts()
    linhas.append({
        'prestação estimada acima de': f'{corte}x o DTI publicado',
        'aprovados': int(contagem.get('aprovado', 0)),
        'negados': int(contagem.get('negado', 0)),
    })
pd.DataFrame(linhas).set_index('prestação estimada acima de')

,aprovados,negados
prestação estimada acima de,,
1.5x o DTI publicado,667,137
2.0x o DTI publicado,600,105
3.0x o DTI publicado,554,81


**Onde olhar:** as três linhas de corte.

**O que se vê:** no grupo de ajuste, com o corte de 2 vezes, são 705 pedidos
incoerentes (600 aprovados e 105 negados). Mudar o corte muda pouco a contagem. Com
1,5 vez são 667 aprovados e 137 negados, e com 3 vezes, 554 e 81.

**O que significa:** saem da base inteira os 10.817 pedidos com renda zero ou
negativa e os 2.244 incoerentes (contagem abaixo). Corrigir a renda seria trocar o
dado do banco por um palpite. O corte em 2 vezes fica no meio, e a escolha entre os
três não muda a conclusão. A renda vazia (24.385 aprovados e 230 negados no ajuste,
nesta altura) fica fora dessa regra. A maior parte
sai junto com o DTI vazio, na seção 4.3, e o resto é tratado na 4.6.

In [23]:
remover(hmda['renda'] <= 0, 'renda zero ou negativa')
remover((hmda['renda'] > 0) & (hmda['razao_renda'] > 2), 'renda incoerente com o DTI')

renda zero ou negativa: saem 10.817 pedidos
  originado: 7.124
  negado: 2.153
  aprovado, não aceito: 1.540
ficam na base: 1.064.544
renda incoerente com o DTI: saem 2.244 pedidos
  originado: 1.828
  negado: 369
  aprovado, não aceito: 47
ficam na base: 1.062.300


### 4.2 DTI ou CLTV "Exempt"

**Pergunta:** o que fazer com o DTI ou o CLTV marcado como "Exempt"?

Banco que faz poucos financiamentos por ano pode ser dispensado de publicar alguns
campos, e o HMDA marca esses campos como "Exempt". A célula conta os pedidos e mede,
em cada banco que usa a marca, que parte dos seus pedidos vem assim. Essa parte
decide o tratamento:

- se o banco marca todos os pedidos, a marca é do banco e não tem a ver com o
  pedido, e tirar esses pedidos não favorece nenhum tipo de caso;
- se marca só alguns, é preciso desconfiar, porque o banco pode estar escondendo o
  DTI justamente dos casos mais difíceis, e tirar esses pedidos distorceria a
  comparação.

In [24]:
base = hmda[na_base()]
exempt = (base['debt_to_income_ratio'] == 'Exempt') | (
    base['loan_to_value_ratio'] == 'Exempt'
)
print('pedidos com DTI ou CLTV Exempt:', base.loc[exempt, 'decisao'].value_counts().to_dict())
bancos = base.loc[exempt, 'lei'].unique()
print('bancos:', len(bancos))
parcela = []
for lei in bancos:
    do_banco = exempt[base['lei'] == lei]
    parcela.append(do_banco.mean())
parcela = pd.Series(parcela)
print('% dos pedidos do banco que vêm Exempt, nesses bancos:')
print((parcela * 100).describe().round(1))

pedidos com DTI ou CLTV Exempt: {'aprovado': 2377}
bancos: 125


% dos pedidos do banco que vêm Exempt, nesses bancos:
count    125.0
mean      98.5
std       11.5
min        0.8
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64


**Onde olhar:** a contagem e a mediana (linha "50%") da parcela por banco.

**O que se vê:** são 2.377 pedidos, todos aprovados, de 125 bancos. Na mediana, o
banco marca "Exempt" em 100% dos seus pedidos.

**O que significa:** vale o primeiro cenário. A marca vem do banco inteiro, e tirar
esses pedidos não favorece nenhum tipo de caso. Sem DTI ou CLTV não há o que
comparar, e saem os 2.377. São todos aprovados porque os negados com "Exempt" já
saíram na seção 3, que exige o campo do motivo publicado.

In [25]:
exempt = (hmda['debt_to_income_ratio'] == 'Exempt') | (
    hmda['loan_to_value_ratio'] == 'Exempt'
)
remover(exempt, 'DTI ou CLTV Exempt')

DTI ou CLTV Exempt: saem 2.377 pedidos
  originado: 2.326
  aprovado, não aceito: 51


ficam na base: 1.059.923


### 4.3 Sem DTI

**Pergunta:** o que fazer com o DTI vazio?

Sem DTI, o pedido não cai em nenhuma faixa, e toda a comparação do projeto é feita por
faixa de DTI. A tabela mostra em que finalidades estão esses pedidos.

In [26]:
base = hmda[no_ajuste()]
sem_dti = base[~base['tem_dti']]
pd.crosstab(sem_dti['finalidade'], sem_dti['decisao'], margins=True,
            margins_name='total')

decisao,aprovado,negado,total
finalidade,,,
compra,2647,42,2689
não se aplica,21,0,21
outro,1754,1,1755
refinanciamento com saque,719,106,825
refinanciamento sem saque,27382,112,27494
reforma,294,0,294
total,32817,261,33078


**Onde olhar:** a linha "refinanciamento sem saque" e a coluna "negado".

**O que se vê:** no grupo de ajuste, são 33.078 pedidos sem DTI, quase todos
aprovados (32.817 contra 261 negados). Desses, 27.494 são refinanciamentos sem saque.

**O que significa:** o padrão aponta para o Streamline, o refinanciamento
simplificado do FHA. Ele só vale para quem já tem financiamento FHA e troca por outro
sem tirar dinheiro, e o banco pode aprová-lo sem analisar renda nem dívidas. Sem essa
análise, não existe DTI para publicar. Isso explica a tabela, em que o DTI vazio se
concentra no refinanciamento sem saque e quase todos esses pedidos foram aprovados.
O notebook 01 (seção 1.5) já tinha encontrado outro sinal da mesma modalidade, o CLTV
vazio em 41,6% dos refinanciamentos sem saque, porque o Streamline também dispensa a
avaliação do imóvel.

No lugar do DTI, o Streamline exige outras coisas, como pagamentos em dia no
financiamento atual e um benefício concreto para o cliente, por exemplo juros
menores. Nada disso está no HMDA, que também não marca quais refinanciamentos são
Streamline. A norma em código não teria como dar uma resposta esperada justa a esses
pedidos, e sem DTI não há faixa onde compará-los. Saem os 109.220.

In [27]:
remover(~hmda['tem_dti'], 'sem DTI')

sem DTI: saem 109.220 pedidos
  originado: 92.989
  aprovado, não aceito: 15.412
  negado: 819
ficam na base: 950.703


### 4.4 Ocupação e posição da hipoteca

**Pergunta:** todos os pedidos que restam são do tipo que a norma analisada cobre?

O Handbook só aceita imóvel de investimento em exceções como o Streamline
("Investment Properties are not eligible for FHA insurance", p. 186), e a segunda
residência depende de aprovação do HUD. O financiamento FHA também precisa ser a
primeira hipoteca do imóvel ("Valid First Liens", p. 156). As tabelas contam a
ocupação e a posição da hipoteca.

In [28]:
base = hmda[no_ajuste()]
ocupacao = pd.crosstab(base['occupancy_type'].map(OCCUPANCY), base['decisao'],
                       margins=True, margins_name='total')
ocupacao = ocupacao.rename_axis(index='ocupação', columns='decisão do banco')
print('Ocupação do imóvel')
display(ocupacao)
hipoteca = pd.crosstab(base['lien_status'].map(LIEN), base['decisao'],
                       margins=True, margins_name='total')
hipoteca = hipoteca.rename_axis(index='posição da hipoteca',
                                columns='decisão do banco')
print('Posição da hipoteca')
display(hipoteca)

Ocupação do imóvel


decisão do banco,aprovado,negado,total
ocupação,,,
investimento,118,29,147
residência principal,249176,36126,285302
segunda residência,6,14,20
total,249300,36169,285469


Posição da hipoteca


decisão do banco,aprovado,negado,total
posição da hipoteca,,,
hipoteca subordinada,139,49,188
primeira hipoteca,249161,36120,285281
total,249300,36169,285469


**Onde olhar:** as linhas "investimento", "segunda residência" e "hipoteca
subordinada".

**O que se vê:** são poucos pedidos no grupo de ajuste, 147 de investimento (118
aprovados), 20 de segunda residência e 188 em hipoteca subordinada.

**O que significa:** esses pedidos estão fora do escopo da norma que o projeto usa,
que trata de residência principal com o FHA como primeira hipoteca. Por isso saem
todos, qualquer que seja o banco ou a decisão. Na base inteira saem 1.153, contados
uma vez só, porque alguns pedidos estão em dois grupos ao mesmo tempo (um imóvel de
investimento em hipoteca subordinada, por exemplo). Ocupação e posição da
hipoteca continuam no perfil do caso, porque o memorando do agente confere esses
campos.

In [29]:
fora_do_escopo = (hmda['occupancy_type'] != '1') | (hmda['lien_status'] != '1')
remover(fora_do_escopo, 'fora de residência principal em primeira hipoteca')

fora de residência principal em primeira hipoteca: saem 1.153 pedidos
  originado: 839
  negado: 293
  aprovado, não aceito: 21


ficam na base: 949.550


### 4.5 Finalidade sem teto na tabela

**Pergunta:** a norma em código consegue avaliar a garantia em todas as finalidades?

A tabela de tetos de CLTV (`CLTV_CAP`, em `src/rules.py`) tem três finalidades:
compra (96,5%), refinanciamento sem saque (97,75%) e refinanciamento com saque (80%).
O HMDA tem ainda reforma, "outro" e "não se aplica". As duas últimas não dizem
qual operação foi feita, e a reforma não tem teto na tabela do projeto. A tabela
mostra, por finalidade, quantos pedidos são negados e quantos desses negados citam
garantia.

In [30]:
base = hmda[no_ajuste()]
linhas = []
for codigo, nome in LOAN_PURPOSE.items():
    da_finalidade = base[base['loan_purpose'] == codigo]
    negados = da_finalidade[da_finalidade['decisao'] == 'negado']
    cita_garantia = (negados[MOTIVOS] == '4').any(axis=1).sum()
    linhas.append({
        'finalidade': nome,
        'teto na tabela': codigo in CLTV_CAP,
        'pedidos': len(da_finalidade),
        'negados': len(negados),
        '% negados': round(100 * len(negados) / len(da_finalidade), 1),
        '% dos negados que citam garantia': round(100 * cita_garantia / len(negados), 1),
    })
pd.DataFrame(linhas).set_index('finalidade')

,teto na tabela,pedidos,negados,% negados,% dos negados que citam garantia
finalidade,,,,,
compra,True,208437,15460,7.4,21.7
reforma,False,737,205,27.8,64.4
refinanciamento sem saque,True,14653,3534,24.1,41.4
refinanciamento com saque,True,58799,15993,27.2,50.7
outro,False,2375,862,36.3,59.0
não se aplica,False,116,23,19.8,60.9


**Onde olhar:** a coluna "% dos negados que citam garantia", comparando as linhas sem
teto com as outras.

**O que se vê:** no grupo de ajuste, nas três finalidades sem teto, os negados citam
garantia em 64,4% (reforma), 59,0% ("outro") e 60,9% ("não se aplica"). Nas
finalidades com teto, em 21,7% (compra), 41,4% (sem saque) e 50,7% (saque).

**O que significa:** nessas finalidades a garantia pesa muito na negativa, e a norma em
código não tem teto para conferir. Ela olharia só o DTI e responderia "aprovar" num
pedido que o banco negou por garantia. Na Evaluation, isso contaria como erro do
agente e da norma, quando o que falta é a regra. Saem as três finalidades inteiras,
aprovados e negados, 10.647 pedidos.

In [31]:
sem_teto = ~hmda['loan_purpose'].isin(list(CLTV_CAP))
remover(sem_teto, 'finalidade sem teto na tabela')

finalidade sem teto na tabela: saem 10.647 pedidos
  originado: 6.727
  negado: 3.664
  aprovado, não aceito: 256
ficam na base: 938.903


### 4.6 O que fica sem regra especial

**Pergunta:** que problemas do dado ficam na base, e com que tratamento?

Nem todo campo vazio ou estranho tira o pedido. Quando a falta é da base e não do
cliente (o banco tinha a informação e só não a publicou), o pedido fica e o perfil
mostra "não informado", como já acontece com score, PTI e reserva no HMDA. Três casos
passam por aqui: CLTV vazio, renda vazia e o código 6 do sistema automático. O
primeiro é o CLTV vazio com DTI publicado.

In [32]:
base = hmda[no_ajuste()]
sem_cltv = base[~base['tem_cltv']]
print('DTI publicado e CLTV vazio, por finalidade')
pd.crosstab(sem_cltv['finalidade'], sem_cltv['decisao'], margins=True,
            margins_name='total')

DTI publicado e CLTV vazio, por finalidade


decisao,aprovado,negado,total
finalidade,,,
compra,1755,252,2007
refinanciamento com saque,194,64,258
refinanciamento sem saque,77,41,118
total,2026,357,2383


**Onde olhar:** a coluna "total" e a linha "compra".

**O que se vê:** no grupo de ajuste, 2.383 pedidos têm DTI e não têm CLTV, 2.026
aprovados e 357 negados, a maioria compras (2.007).

**O que significa:** com o DTI publicado, o pedido tem faixa e pode ser comparado. A
falta do CLTV parece ser da base, e não do cliente. Numa compra, que é a maioria
aqui, o banco precisa saber o valor do imóvel para aprovar, então o mais provável é
que o CLTV só não tenha sido publicado. Os negados que ficam têm todos o DTI
entre os motivos, porque os negados por garantia sem CLTV saíram na seção 3.4. O
motivo de cada negativa tem na base o número que o confirma. Esses pedidos ficam com
"CLTV: não informado" no perfil, e a resposta esperada decide só pelo DTI. Na
Evaluation, a concordância deles sai separada (coluna `cltv_vazio`). Se o agente
suspender o pedido para pedir a avaliação do imóvel, a discordância fica isolada
nesse grupo, e o motivo é conhecido.

**Pergunta:** nos pedidos com DTI e renda vazia, o DTI é confiável?

Mesmo sem a renda publicada, o banco pode ter calculado o DTI com ela. A célula
compara a distribuição de DTI com renda preenchida e com renda vazia e mostra em
quantos bancos a falta aparece.

In [33]:
base = hmda[no_ajuste()].copy()
base['renda'] = base['renda'].isna().map({True: 'vazia', False: 'preenchida'})
print('renda vazia:', base.loc[base['renda'] == 'vazia', 'decisao'].value_counts().to_dict())
for decisao in ['aprovado', 'negado']:
    grupo = base[base['decisao'] == decisao]
    tabela = pd.crosstab(grupo['faixa_dti'], grupo['renda'], normalize='columns')
    tabela = (tabela * 100).round(1).reindex(FAIXAS)
    tabela = tabela.rename_axis(index='faixa de DTI', columns='renda')
    print(f'{decisao}s, faixa de DTI (% da coluna)')
    display(tabela)
negados_vazia = base[(base['renda'] == 'vazia') & (base['decisao'] == 'negado')]
print()
print('negados com renda vazia e DTI entre os motivos:',
      br(negados_vazia['motivo_dti_visivel'].sum()), 'de', br(len(negados_vazia)))
por_banco = base[base['renda'] == 'vazia']['lei'].value_counts()
print('bancos com renda vazia:', len(por_banco))
for lei in por_banco.index[:3]:
    do_banco = base[base['lei'] == lei]
    print(f'  {lei}: {br(por_banco[lei])} de {br(len(do_banco))} pedidos')

renda vazia: {'aprovado': 860, 'negado': 107}
aprovados, faixa de DTI (% da coluna)


renda,preenchida,vazia
faixa de DTI,,
até 43,42.3,69.1
44 a 49,27.1,16.6
50 a 60,30.4,13.5
acima de 60,0.2,0.8


negados, faixa de DTI (% da coluna)


renda,preenchida,vazia
faixa de DTI,,
até 43,20.6,19.6
44 a 49,13.7,12.1
50 a 60,30.5,15.0
acima de 60,35.2,53.3



negados com renda vazia e DTI entre os motivos: 89 de 107
bancos com renda vazia: 174
  549300GKFNPRWNS0GF29: 116 de 4.712 pedidos
  5493004R423PBPM6A173: 91 de 100 pedidos
  5493001WHVQBGRSWEU75: 87 de 505 pedidos


**Onde olhar:** a distribuição por faixa de DTI, renda preenchida contra vazia, e a
lista dos bancos.

**O que se vê:** no grupo de ajuste, 967 pedidos têm DTI e não têm renda (860
aprovados e 107 negados). Nos aprovados com renda vazia, o DTI é mais baixo que o
normal (69,1% até 43%, contra 42,3%). Nos negados, 53,3% estão acima de 60% (contra
35,2%), e 89 dos 107 citam DTI. A renda vazia aparece em 174 bancos, e só um da lista
tem renda vazia em quase tudo (91 de 100 pedidos).

**O que significa:** o DTI desses pedidos combina com a decisão do banco. Os
aprovados têm DTI baixo, os negados têm DTI alto, e quase todos os negados citam o
DTI como motivo. Um DTI calculado errado dificilmente bateria tão bem com a decisão.
A lista dos bancos verifica se a falta vem de um banco só, o que indicaria erro de
preenchimento. Não vem. A renda vazia aparece em 174 bancos, e dos três com mais
casos só um tem o campo vazio em quase tudo (91 de 100 pedidos). O mais
provável é que o banco tinha a renda, usou-a no DTI e só não a publicou. Os pedidos
com renda vazia ficam, com "renda: não informado" no perfil e a coluna `renda_vazia` para
separar o resultado deles na Evaluation. Como o DTI já existe, o prompt da Modeling
precisa evitar que o agente suspenda o pedido só para pedir a renda.

**Pergunta:** o código 6 do campo `aus-1` separa os pedidos de análise manual?

No FHA, quase todo pedido passa primeiro pelo sistema automático do HUD, o TOTAL
Scorecard. Se o sistema aceita, o banco pode aprovar com os limites dele, que passam
de 50% de DTI. Se o sistema manda para o analista, o pedido segue a tabela manual,
que não aprova DTI acima de 50% em nenhuma linha. O projeto trata desse segundo
caminho. O `aus-1` diz qual sistema o banco usou, e o código 6 quer dizer "não se
aplica". À primeira vista, seria o grupo em que o analista decidiu sem o sistema. A
célula compara esse grupo com o resto. Se o código 6 marcasse a análise manual, os
aprovados com ele quase nunca teriam DTI acima de 50%.

In [34]:
print('pedidos com aus-1 = 6 na base inteira:',
      br((hmda.loc[na_base(), 'aus-1'] == '6').sum()))
base = hmda[no_ajuste()].copy()
base['aus-1 = 6'] = (base['aus-1'] == '6').map({True: 'sim', False: 'não'})
por_decisao = pd.crosstab(base['aus-1 = 6'], base['decisao'], margins=True,
                          margins_name='total')
por_decisao = por_decisao.rename_axis(index='aus-1 = 6',
                                      columns='decisão do banco')
print('Código 6 no aus-1, por decisão')
display(por_decisao)
aprovados = base[base['decisao'] == 'aprovado']
tabela = pd.crosstab(aprovados['faixa_dti'], aprovados['aus-1 = 6'],
                     normalize='columns')
tabela = (tabela * 100).round(1).reindex(FAIXAS)
tabela = tabela.rename_axis(index='faixa de DTI', columns='aus-1 = 6')
print('aprovados, faixa de DTI (% da coluna)')
display(tabela)
parcela = {}
for lei, do_banco in aprovados.groupby('lei'):
    parcela[lei] = (do_banco['aus-1'] == '6').mean()
parcela = pd.Series(parcela)
casos = aprovados[aprovados['aus-1'] == '6'].groupby('lei').size()
quase_todos = parcela[parcela >= 0.95].index
print()
print('aprovados com aus-1 = 6:', br(casos.sum()),
      '| em bancos que marcam 6 em 95% ou mais dos aprovados:',
      br(casos.reindex(quase_todos).fillna(0).sum()))

pedidos com aus-1 = 6 na base inteira: 24.385


Código 6 no aus-1, por decisão


decisão do banco,aprovado,negado,total
aus-1 = 6,,,
não,243746,30855,274601
sim,3156,4132,7288
total,246902,34987,281889


aprovados, faixa de DTI (% da coluna)


aus-1 = 6,não,sim
faixa de DTI,,
até 43,42.1,63.1
44 a 49,27.2,18.4
50 a 60,30.6,10.5
acima de 60,0.1,8.0



aprovados com aus-1 = 6: 3.156 | em bancos que marcam 6 em 95% ou mais dos aprovados: 252


**Onde olhar:** a coluna "negado" da primeira tabela, a coluna "sim" da segunda e a
última linha.

**O que se vê:** na base inteira, 24.385 pedidos têm o código 6. No grupo de ajuste
são 7.288, e 4.132 deles foram negados (57%, contra 11% nos outros). Nos aprovados
com o código, o DTI de 50% a 60% é raro (10,5% contra 30,6%), como seria na análise
manual, mas 8,0% estão acima de 60% (contra 0,1%), o que a análise manual não
aprovaria. Dos 3.156 aprovados com o código 6, 252 vêm de bancos que marcam 6 em 95%
ou mais dos seus aprovados.

**O que significa:** o código 6 não separa a análise manual. Ele junta pedidos com
cara de análise manual (pouco DTI de 50% a 60%), aprovações acima de 60%, que nenhuma
linha da tabela manual permite, e bancos que preenchem 6 em quase tudo. Além disso, o
pedido que o sistema manda para o analista continua registrado com o código do
sistema, e não com o 6. O campo que separaria a análise manual é o resultado do
sistema (aceito ou mandado ao analista), e o HMDA público não o publica. Por isso
nenhum grupo do HMDA representa só a análise manual, e o projeto compara a norma
manual com decisões que, em boa parte, vieram do sistema automático. Essa é a linha
que vai para a tabela de limitações. O campo também fica fora do perfil do agente,
porque daria uma pista de como o banco analisou o pedido.

**Pergunta:** o CLTV publicado já inclui o seguro inicial do FHA?

O teto de CLTV de cada finalidade vale para o empréstimo antes do seguro inicial do
FHA (UFMIP, 1,75% do empréstimo), que o cliente pode somar à dívida por cima:
"restrictions to mortgage amounts and LTVs are based upon the amount prior to the
financing of the UFMIP" (II.A.2.iii). Já a Regulation C pede o CLTV "relied on in
making the credit decision" e diz que "does not require a particular calculation
method" (comentário 4(a)(24)). Cada banco pode, então, publicar o CLTV com o seguro
somado ou sem ele, e o HMDA não tem campo que diga qual.

Isso muda a leitura do número. Um cliente de refinanciamento sem saque que pegou o
máximo, 97,75%, aparece com 97,75 num banco que publica sem o seguro e com 99,46
(97,75 vezes 1,0175) num banco que publica com ele. Um CLTV de 98,5 quer dizer coisas
opostas nos dois bancos. No primeiro, o empréstimo passou do teto. No segundo, fica
abaixo dele e cumpre a norma.

A pista está em onde os CLTVs se amontoam. Quem pega o máximo aparece exatamente no
teto, se o banco publica sem o seguro, ou exatamente no teto vezes 1,0175, se publica
com ele. A primeira célula conta os pedidos nesses dois valores.

In [35]:
base = hmda[na_base()].copy()
no_teto = []
no_teto_com_seguro = []
for finalidade, cltv in zip(base['loan_purpose'], base['cltv']):
    teto = CLTV_CAP[finalidade]
    teto_seguro = teto * (1 + UFMIP / 100)
    # O HMDA arredonda o teto com o seguro de jeitos diferentes (98,188, 98,19).
    no_teto.append(abs(cltv - teto) <= 0.005)
    no_teto_com_seguro.append(abs(cltv - teto_seguro) <= 0.015)
base['no_teto'] = no_teto
base['no_teto_com_seguro'] = no_teto_com_seguro

picos = base.groupby('finalidade')[['no_teto', 'no_teto_com_seguro']].sum()
picos = picos.rename(columns={'no_teto': 'CLTV no teto',
                              'no_teto_com_seguro': 'CLTV no teto com o seguro'})
print('Pedidos com o CLTV exatamente no teto e no teto com o seguro')
display(picos)

Pedidos com o CLTV exatamente no teto e no teto com o seguro


,CLTV no teto,CLTV no teto com o seguro
finalidade,,
compra,396530,14448
refinanciamento com saque,58186,2557
refinanciamento sem saque,2200,43


**Onde olhar:** as duas colunas, finalidade a finalidade.

**O que se vê:** nas três finalidades, o teto sem o seguro junta muito mais pedidos
que o teto com o seguro: 396.530 contra 14.448 na compra, 58.186 contra 2.557 no
saque e 2.200 contra 43 no sem saque.

**O que significa:** na maior parte dos pedidos, quem pegou o máximo aparece
exatamente no teto, ou seja, o CLTV publicado não traz o seguro somado. O pico no
teto com o seguro também existe, então há bancos que publicam com ele.

O jeito de publicar é uma escolha do banco, que vale para todos os seus pedidos. A
célula seguinte conta, banco a banco, em qual dos dois valores ele tem mais pedidos. A conta não usa a decisão do
banco, só onde os CLTVs se amontoam. O banco com mais pedidos no teto com o seguro
fica como "com o seguro"; os demais, inclusive os que não têm pedido em nenhum dos
dois valores, ficam como "sem o seguro", que é o jeito da maioria e o mais cauteloso,
porque não abre a faixa do seguro.

In [36]:
por_banco = base.groupby('lei')[['no_teto', 'no_teto_com_seguro']].sum()


def picos_do_banco(linha):
    """Em quais dos dois valores o banco tem pedidos."""
    tem_teto = linha['no_teto'] > 0
    tem_teto_seguro = linha['no_teto_com_seguro'] > 0
    if tem_teto and tem_teto_seguro:
        return 'os dois'
    if tem_teto:
        return 'só no teto'
    if tem_teto_seguro:
        return 'só no teto com o seguro'
    return 'nenhum'


def jeito_do_banco(linha):
    """Pico maior no teto com o seguro: publica com ele; senão, sem ele."""
    if linha['no_teto_com_seguro'] > linha['no_teto']:
        return 'com o seguro'
    return 'sem o seguro'


por_banco['picos'] = por_banco.apply(picos_do_banco, axis=1)
por_banco['cltv_publicado'] = por_banco.apply(jeito_do_banco, axis=1)
bancos = pd.crosstab(por_banco['picos'], por_banco['cltv_publicado'],
                     margins=True, margins_name='total')
bancos = bancos.rename_axis(index='pedidos nos picos',
                            columns='CLTV publicado')
print('Bancos por pico e por jeito de publicar o CLTV')
display(bancos)

base['cltv_publicado'] = base['lei'].map(por_banco['cltv_publicado'])
na_faixa = []
for finalidade, cltv in zip(base['loan_purpose'], base['cltv']):
    na_faixa.append(in_ufmip_band(finalidade, cltv))
faixa = base[na_faixa]
faixa = faixa[faixa['grupo'] == 'ajuste']
tabela = pd.crosstab([faixa['finalidade'], faixa['cltv_publicado']],
                     faixa['decisao'])
tabela['total'] = tabela['aprovado'] + tabela['negado']
tabela['% negado'] = (100 * tabela['negado'] / tabela['total']).round(1)
tabela = tabela.rename_axis(index=['finalidade', 'CLTV publicado'],
                            columns='decisão do banco')
print('Pedidos entre o teto e o teto com o seguro, por jeito do banco')
display(tabela)

Bancos por pico e por jeito de publicar o CLTV


CLTV publicado,com o seguro,sem o seguro,total
pedidos nos picos,,,
nenhum,0,90,90
os dois,24,78,102
só no teto,0,1004,1004
só no teto com o seguro,25,0,25
total,49,1172,1221


Pedidos entre o teto e o teto com o seguro, por jeito do banco


decisão do banco                          aprovado  negado  total  % negado
finalidade                CLTV publicado                                   
compra                    com o seguro        4198     503   4701      10.7
                          sem o seguro        1229     216   1445      14.9
refinanciamento com saque com o seguro         518     366    884      41.4
                          sem o seguro          25     148    173      85.5
refinanciamento sem saque com o seguro          29      10     39      25.6
                          sem o seguro          23     135    158      85.4

**Onde olhar:** na primeira tabela, a linha "total"; na segunda, a coluna "% negado".

**O que se vê:** 1.004 bancos têm pedidos só no teto, 25 só no teto com o seguro, 102
nos dois e 90 em nenhum. Pelo pico maior, 49 bancos publicam com o seguro e 1.172
sem ele. Entre o teto e o teto com o seguro, no grupo de ajuste, os bancos que
publicam sem o seguro negaram 85,5% dos refinanciamentos com saque e 85,4% dos sem
saque, contra 41,4% e 25,6% nos bancos que publicam com o seguro. Na compra, 14,9%
contra 10,7%.

**O que significa:** nos bancos que publicam sem o seguro, um CLTV nessa faixa quer
dizer empréstimo acima do teto, e os bancos agiram assim, negando quase todos os
refinanciamentos. Na compra a diferença é menor, porque a compra acima do teto ainda
pode ser aprovada com segundo financiamento de órgão público, que não tem teto. A
classificação pela maioria é um palpite nos 102 bancos com os dois picos e nos 90
sem pico, o que vai para a tabela de limitações.

**Decisão:** cada pedido recebe a coluna `cltv_publicado`, com o jeito do seu banco.
Nos bancos "sem o seguro", a faixa entre o teto e o teto com o seguro conta como
acima do teto (seção 5). Nenhum pedido sai. O perfil do caso diz como o CLTV foi
publicado (seção 8), porque o analista real conhece o valor do empréstimo com e sem
o seguro.

In [37]:
hmda['cltv_publicado'] = hmda['lei'].map(por_banco['cltv_publicado'])
# Banco sem pedido na base não tem pico: fica como os demais sem pico.
hmda['cltv_publicado'] = hmda['cltv_publicado'].fillna('sem o seguro')
hmda.loc[na_base(), 'cltv_publicado'].value_counts()

cltv_publicado
sem o seguro    905714
com o seguro     33189
Name: count, dtype: int64

**O que se vê:** 905.714 pedidos da base ficam como "sem o seguro" e 33.189 como "com
o seguro".

### 4.7 Funil do HMDA

**Pergunta:** quanto saiu em cada etapa, e o que sobrou em cada grupo?

A primeira tabela junta todas as remoções do notebook, na ordem em que foram
aplicadas. As colunas são os dois grupos sorteados na seção 1, o de ajuste, onde o
prompt é testado, e o da nota final, que roda uma vez só, na Evaluation. Se as
remoções não favorecem nenhum grupo, cada linha fica perto de 30% no ajuste e 70% na
nota final. A segunda tabela mostra a decisão do banco nos pedidos que ficam no
grupo de ajuste.

Legenda das linhas do funil:

| Linha do funil | O que entra | Seção |
|---|---|---|
| **Desfechos sem gabarito** | | |
| sem decisão de crédito | comprado de terceiro, retirado pelo cliente, pré aprovação aprovada e não aceita, pré aprovação negada | 2.1 |
| fechado por incompletude | o cliente não entregou um documento e o banco encerrou o pedido sem decidir | 2.2 |
| **Negativas sem motivo visível** | | |
| negado por cadastro incompleto | negativa por documento faltante (código 7), o mesmo acontecimento do fechado por incompletude | 3.1 |
| motivo invisível sem motivo visível | 1º motivo histórico de crédito, emprego ou informação não verificável, sem DTI nem garantia em nenhum dos quatro motivos | 3.2 |
| grupo outros sem motivo visível | 1º motivo "outro", recursos insuficientes, isento ou seguro hipotecário negado, sem DTI nem garantia nos quatro motivos | 3.3 |
| motivo visível sem o campo publicado | negado por DTI ou garantia, mas com o DTI ou o CLTV vazio na base | 3.4 |
| **Qualidade do dado** | | |
| renda zero ou negativa | renda com a qual o DTI não tem como existir | 4.1 |
| renda incoerente com o DTI | prestação estimada acima de 2 vezes o topo do DTI publicado | 4.1 |
| DTI ou CLTV Exempt | banco dispensado de publicar esses campos | 4.2 |
| sem DTI | DTI vazio, na maior parte provável Streamline | 4.3 |
| fora de residência principal em primeira hipoteca | investimento, segunda residência ou hipoteca subordinada | 4.4 |
| finalidade sem teto na tabela | reforma, "outro" e "não se aplica" | 4.5 |
| **fica na base** | pedidos que seguem para as marcações da seção 5 | |

In [38]:
ORDEM = [
    'sem decisão de crédito',
    'fechado por incompletude',
    'negado por cadastro incompleto',
    'motivo invisível sem motivo visível',
    'grupo outros sem motivo visível',
    'motivo visível sem o campo publicado',
    'renda zero ou negativa',
    'renda incoerente com o DTI',
    'DTI ou CLTV Exempt',
    'sem DTI',
    'fora de residência principal em primeira hipoteca',
    'finalidade sem teto na tabela',
    'fica na base',
]
etapa = hmda['sai_por'].replace('', 'fica na base')
funil = pd.crosstab(etapa, hmda['grupo'], margins=True, margins_name='total')
funil = funil.reindex(ORDEM + ['total'])
funil

grupo,ajuste,nota final,total
sai_por,,,
sem decisão de crédito,222643,520976,743619
fechado por incompletude,30401,71617,102018
negado por cadastro incompleto,11829,27612,39441
motivo invisível sem motivo visível,18826,43974,62800
grupo outros sem motivo visível,10872,25206,36078
motivo visível sem o campo publicado,1076,2541,3617
renda zero ou negativa,3250,7567,10817
renda incoerente com o DTI,705,1539,2244
DTI ou CLTV Exempt,732,1645,2377


In [39]:
fica = hmda[no_ajuste()]
fica['decisao'].value_counts().rename_axis('decisão do banco (ajuste)').to_frame()

,count
decisão do banco (ajuste),
aprovado,246902
negado,34987


**Onde olhar:** a linha "fica na base" e a segunda tabela.

**O que se vê:** dos 2.062.934 pedidos, ficam 938.903. O grupo de ajuste fica com
281.889 (246.902 aprovados e 34.987 negados) e o da nota final com 657.014. A
maior saída é a dos pedidos sem decisão de crédito (743.619), e entre as regras de
qualidade, a do DTI vazio (109.220).

**O que significa:** depois de todas as remoções, o grupo de ajuste continua com 30,0%
da base, então nenhuma regra pesou mais para um grupo. É desta base que saem as
marcações da seção 5.

<a id="secao-5"></a>

## 5. HMDA: marcações para a avaliação

**A pergunta.** Quais pedidos precisam de resultado separado na Evaluation, e em que
parte da avaliação cada um é medido?

Daqui em diante nenhum pedido sai. A seção 4 decidiu quem fica na base, e esta seção
decide como cada pedido vai ser corrigido na Evaluation. É como a correção de uma
prova em que nem toda questão tem o mesmo gabarito. Em parte dos pedidos, a decisão
do banco serve de gabarito. Em outra parte, o banco viu informações que a base não
mostra, e o gabarito passa a ser o que a norma manda fazer. Numa terceira parte, a
decisão veio do sistema automático, fora da análise manual, e o pedido fica fora da
nota. A coluna `onde_entra` guarda essa divisão:

| Parte | Pedidos | Gabarito | Por quê |
|---|---|---|---|
| concordância com o banco | DTI até 43% ou acima de 60% | decisão do banco | a norma responde direto, aprovando até 43% e negando acima de 60% |
| contra a norma (fator compensatório) | DTI de 44% a 49% | suspender pedindo a comprovação | a norma pede um fator compensatório que a base não mostra |
| contra a norma (segundo financiamento) | compra com CLTV acima do teto (98,2% se o banco publica o CLTV com o seguro, 96,5% se publica sem) | suspender pedindo a comprovação | a norma só aceita com segundo financiamento de órgão público ou entidade aprovada pelo HUD, que a base não mostra |
| relatório à parte | DTI de 50% a 60% | nenhum, fica fora da nota | o HMDA junta o 50%, permitido com dois fatores, com valores que a análise manual não aprova, e as aprovações dessa faixa vêm do sistema automático |

Alguns pedidos cabem em duas partes ao mesmo tempo. O que fazer com eles é decidido
mais abaixo, depois das marcações, com os números na mão.

As demais colunas servem para fatiar o resultado e achar onde o agente erra. Se ele
errar muito nos pedidos com `cltv_vazio`, por exemplo, o erro vem da falta do CLTV, e
não de uma falha geral. São elas:

- `subfaixa_ate_43` divide o até 43% em até 35% e 36% a 43%. O HMDA não publica o
  PTI, e a linha "sem fator" da tabela manual exige PTI até 31%. Na NSMO, só 3,2% dos
  casos até 35% passam desse PTI, contra cerca de 1 em 5 de 36% a 43% (notebook 01,
  seção 7.1).
- `cltv_90` separa o CLTV até 90% do CLTV acima. Acima de 90%, a norma exige score de
  580, que o HMDA não publica.
- `situacao_cltv` compara o CLTV com o teto da finalidade. O teto vale antes do seguro
  inicial do FHA (UFMIP, Upfront Mortgage Insurance Premium, 1,75% do empréstimo), que
  o cliente pode somar à dívida (II.A.2.iii). Com o seguro somado, o teto da compra
  passa de 96,5% para 98,2%, o do saque de 80% para 81,4% e o do sem saque de 97,75%
  para 99,5%. Entre o teto e o teto com o seguro, o pedido cumpre a norma só quando o
  banco publica o CLTV com o seguro (`cltv_publicado`, seção 4.6). Nos demais bancos,
  essa faixa já está acima do teto.
- `seguro_financiado` marca essa faixa em todos os bancos, pelo valor do CLTV, e a
  divide em duas. "Cravado no teto com seguro" é o pedido com CLTV exatamente no teto
  vezes 1,0175, sinal de que o cliente financiou o máximo permitido mais o seguro. Os
  demais ficam como "resto da faixa".
- `cltv_vazio`, `renda_vazia` e `aus_6` marcam os casos da seção 4.6.

In [40]:
casos = hmda[na_base()].copy()


def subfaixa_ate_43(publicado):
    """Divide o até 43% em até 35% e 36% a 43% (decisão do PTI)."""
    if publicado in ('<20%', '20%-<30%', '30%-<36%'):
        return 'até 35'
    if publicado.isdigit() and int(publicado) <= 43:
        return '36 a 43'
    return ''


def cltv_90(cltv):
    """CLTV até 90% ou acima (acima de 90% a norma pede score de 580)."""
    if pd.isna(cltv):
        return 'não informado'
    if cltv <= 90:
        return 'até 90'
    return 'acima de 90'


casos['subfaixa_ate_43'] = casos['debt_to_income_ratio'].map(subfaixa_ate_43)
casos['cltv_90'] = casos['cltv'].map(cltv_90)

situacoes = []
for finalidade, cltv, publicado in zip(
    casos['loan_purpose'], casos['cltv'], casos['cltv_publicado']
):
    com_seguro = publicado == 'com o seguro'
    situacoes.append(cltv_situation(finalidade, cltv, None, com_seguro))
casos['situacao_cltv'] = situacoes

# A faixa do seguro é marcada pelo valor do CLTV, em todos os bancos, para o
# resultado dela poder ser separado na Evaluation.
na_faixa = []
for finalidade, cltv in zip(casos['loan_purpose'], casos['cltv']):
    na_faixa.append(in_ufmip_band(finalidade, cltv))
seguro = pd.Series(na_faixa, index=casos.index)
casos['seguro_financiado'] = ''
casos.loc[seguro, 'seguro_financiado'] = 'resto da faixa'
# Cravado = teto x 1,0175 (98,19 na compra, 81,4 no saque, 99,46 sem saque). O
# HMDA arredonda de jeitos diferentes (98,188, 98,189, 81,39), daí a tolerância.
for finalidade, teto in CLTV_CAP.items():
    alvo = teto * (1 + UFMIP / 100)
    perto = (casos['cltv'] - alvo).abs() <= 0.015
    cravado = seguro & (casos['loan_purpose'] == finalidade) & perto
    casos.loc[cravado, 'seguro_financiado'] = 'cravado no teto com seguro'

casos['cltv_vazio'] = ~casos['tem_cltv']
casos['renda_vazia'] = casos['renda'].isna()
casos['aus_6'] = casos['aus-1'] == '6'

**Pergunta:** o que fazer com o pedido que cabe em duas partes?

A tabela do início da seção separa as partes pela faixa de DTI e pela situação do CLTV, mas os
dois critérios podem apontar para lados diferentes. Um refinanciamento com DTI de
47% e CLTV acima do teto cabe no fator compensatório pelo DTI e numa negativa por
garantia pelo CLTV. Uma compra com DTI acima de 60% e CLTV acima do teto cabe na
negativa por DTI e no segundo financiamento a confirmar. A tabela abaixo conta esses
pedidos e mostra o que a norma em código responde em cada um, sem olhar a decisão
do banco. Em seguida vêm quatro pedidos sorteados.

In [41]:
refi_acima_do_teto = (casos['faixa_dti'] == '44 a 49') & (
    casos['situacao_cltv'] == 'acima do teto'
)
compra_acima_de_60 = (casos['faixa_dti'] == 'acima de 60') & (
    casos['situacao_cltv'] == 'segundo financiamento a confirmar'
)
cabe_em_duas = casos[refi_acima_do_teto | compra_acima_de_60].copy()

acoes = []
motivos = []
for dti, finalidade, cltv, publicado in zip(
    cabe_em_duas['debt_to_income_ratio'], cabe_em_duas['loan_purpose'],
    cabe_em_duas['cltv'], cabe_em_duas['cltv_publicado'],
):
    com_seguro = publicado == 'com o seguro'
    resposta = expected_hmda(dti, finalidade, cltv, com_seguro)
    acoes.append(resposta['acao'])
    motivos.append('; '.join(resposta['motivos']))
cabe_em_duas['resposta da norma'] = acoes
cabe_em_duas['motivo'] = motivos

tabela = pd.crosstab(
    [cabe_em_duas['finalidade'], cabe_em_duas['faixa_dti'],
     cabe_em_duas['situacao_cltv']],
    [cabe_em_duas['resposta da norma'], cabe_em_duas['motivo']],
)
tabela = tabela.rename_axis(index=['finalidade', 'faixa de DTI', 'situação do CLTV'],
                            columns=['resposta da norma', 'motivo'])
print('Pedidos que cabem em duas partes, pela resposta da norma em código')
display(tabela)

colunas = ['finalidade', 'debt_to_income_ratio', 'cltv', 'cltv_publicado',
           'resposta da norma', 'motivo']
print('Quatro pedidos sorteados')
display(cabe_em_duas.sample(4, random_state=SEED)[colunas])

Pedidos que cabem em duas partes, pela resposta da norma em código


resposta da norma                                                        negar         
motivo                                                                     DTI garantia
finalidade                faixa de DTI situação do CLTV                                
compra                    acima de 60  segundo financiamento a confirmar  2754        0
refinanciamento com saque 44 a 49      acima do teto                         0     2275
refinanciamento sem saque 44 a 49      acima do teto                         0      562

Quatro pedidos sorteados


,finalidade,debt_to_income_ratio,cltv,cltv_publicado,resposta da norma,motivo
1633869,compra,>60%,100.000,sem o seguro,negar,DTI
44579,refinanciamento sem saque,45,106.346,sem o seguro,negar,garantia
267721,refinanciamento com saque,45,101.538,sem o seguro,negar,garantia
1265036,compra,>60%,99.500,sem o seguro,negar,DTI


**Onde olhar:** as colunas da resposta da norma e do motivo.

**O que se vê:** são 5.591 pedidos, e a norma nega todos. As 2.754 compras com DTI
acima de 60% e CLTV acima do teto são negadas por DTI. Os 2.837 refinanciamentos com
DTI de 44% a 49% e CLTV acima do teto (2.275 com saque e 562 sem saque) são negados
por garantia. Nenhum termina em suspender, e os quatro sorteados mostram o mesmo,
com CLTV de 99,5% a 106%.

**O que significa:** nesses pedidos a norma nega direto, e nenhuma comprovação muda
a resposta. O fator compensatório só compensa DTI, e o segundo financiamento não
resolve um DTI acima de 60%. Se ficassem nas partes contra a norma, o gabarito seria
"suspender pedindo a comprovação", e o agente que negasse, como manda a norma,
levaria erro. Por isso a função `onde_entra` testa as negativas diretas antes das
partes de suspender, e esses pedidos vão para a concordância com o banco, como os
outros que a norma nega.

In [42]:
def onde_entra(faixa, situacao):
    """Em que parte da avaliação o pedido é medido."""
    if faixa == '50 a 60':
        return 'relatório à parte (DTI de 50% a 60%)'
    # A norma nega direto, e nenhuma comprovação muda a resposta.
    if faixa == 'acima de 60':
        return 'concordância com o banco'
    if situacao == 'acima do teto':
        return 'concordância com o banco'
    if faixa == '44 a 49':
        return 'contra a norma (fator compensatório)'
    if situacao == 'segundo financiamento a confirmar':
        return 'contra a norma (segundo financiamento)'
    return 'concordância com o banco'


destinos = []
for faixa, situacao in zip(casos['faixa_dti'], casos['situacao_cltv']):
    destinos.append(onde_entra(faixa, situacao))
casos['onde_entra'] = destinos
pd.crosstab(casos['onde_entra'], casos['grupo'], margins=True, margins_name='total')

grupo,ajuste,nota final,total
onde_entra,,,
concordância com o banco,108109,251987,360096
contra a norma (fator compensatório),70854,164918,235772
contra a norma (segundo financiamento),17319,41194,58513
relatório à parte (DTI de 50% a 60%),85607,198915,284522
total,281889,657014,938903


In [43]:
def com_percentual(tabela):
    """Acrescenta o total da linha e a % de negados."""
    tabela['total'] = tabela['aprovado'] + tabela['negado']
    tabela['% negado'] = (100 * tabela['negado'] / tabela['total']).round(1)
    return tabela.rename_axis(columns='decisão do banco')


ajuste = casos[casos['grupo'] == 'ajuste']
situacao = pd.crosstab(ajuste['situacao_cltv'], ajuste['decisao'])
situacao = com_percentual(situacao).rename_axis(index='situação do CLTV')
print('Situação do CLTV contra o teto da finalidade')
display(situacao)

com_seguro = ajuste[ajuste['seguro_financiado'] != '']
seguro = pd.crosstab([com_seguro['finalidade'], com_seguro['cltv_publicado'],
                      com_seguro['seguro_financiado']],
                     com_seguro['decisao'])
seguro = com_percentual(seguro).rename_axis(
    index=['finalidade', 'CLTV publicado', 'faixa do seguro']
)
print('Faixa do seguro financiado, por finalidade e jeito do banco')
display(seguro)

Situação do CLTV contra o teto da finalidade


decisão do banco,aprovado,negado,total,% negado
situação do CLTV,,,,
acima do teto,221,4037,4258,94.8
dentro do teto,203474,26618,230092,11.6
não informado,2026,357,2383,15.0
segundo financiamento a confirmar,36436,3096,39532,7.8
seguro financiado,4745,879,5624,15.6


Faixa do seguro financiado, por finalidade e jeito do banco


decisão do banco                                                     aprovado  negado  total  % negado
finalidade                CLTV publicado faixa do seguro                                              
compra                    com o seguro   cravado no teto com seguro      3703     449   4152      10.8
                                         resto da faixa                   495      54    549       9.8
                          sem o seguro   cravado no teto com seguro       193      18    211       8.5
                                         resto da faixa                  1036     198   1234      16.0
refinanciamento com saque com o seguro   cravado no teto com seguro       432     319    751      42.5
                                         resto da faixa                    86      47    133      35.3
                          sem o seguro   cravado no teto com seguro        12       7     19      36.8
                                         resto da faixa                    13     141    154      91.6
refinanciamento sem saque com o seguro   cravado no teto com seguro        11       2     13      15.4
                                         resto da faixa                    18       8     26      30.8
                          sem o seguro   cravado no teto com seguro         2       2      4      50.0
                                         resto da faixa                    21     133    154      86.4

In [44]:
concordancia = ajuste[ajuste['onde_entra'] == 'concordância com o banco']
por_faixa = pd.crosstab([concordancia['faixa_dti'],
                         concordancia['subfaixa_ate_43']],
                        concordancia['decisao'])
por_faixa = com_percentual(por_faixa).rename_axis(index=['faixa de DTI',
                                                         'subfaixa'])
print('Concordância com o banco, por faixa e subfaixa de DTI')
display(por_faixa)

ate_43 = concordancia[concordancia['faixa_dti'] == 'até 43']
por_cltv = pd.crosstab(ate_43['cltv_90'], ate_43['decisao'])
por_cltv = com_percentual(por_cltv).rename_axis(index='CLTV')
print('Concordância com o banco, DTI até 43%, por CLTV')
display(por_cltv)

Concordância com o banco, por faixa e subfaixa de DTI


decisão do banco       aprovado  negado  total  % negado
faixa de DTI subfaixa                                   
44 a 49                      47     783    830      94.3
acima de 60                 422   12324  12746      96.7
até 43       36 a 43      50411    3476  53887       6.5
             até 35       37541    3105  40646       7.6

Concordância com o banco, DTI até 43%, por CLTV


decisão do banco,aprovado,negado,total,% negado
CLTV,,,,
acima de 90,56214,2251,58465,3.9
até 90,30763,4312,35075,12.3
não informado,975,18,993,1.8


**Onde olhar:** são cinco tabelas. A primeira mostra quantos pedidos caem em cada
parte da avaliação, nos dois grupos. As outras quatro usam só o grupo de ajuste. A
segunda compara o CLTV com o teto da finalidade, e a terceira abre a faixa do seguro
financiado por finalidade. A quarta e a quinta mostram o tamanho dos grupos dentro
da concordância com o banco, por faixa de DTI e por CLTV. Da segunda em diante, a
coluna "% negado" diz quanto de cada linha o banco negou.

**O que se vê:**

- Partes da avaliação: 360.096 pedidos na concordância com o banco, 235.772 no fator
  compensatório, 58.513 no segundo financiamento e 284.522 no relatório à parte.
  Cada parte fica perto de 30% no ajuste.
- Situação do CLTV: acima do teto, o banco negou 94,8% dos pedidos. Dentro do teto,
  negou 11,6%; na faixa do seguro financiado, 15,6%; no segundo financiamento a
  confirmar, 7,8%.
- Seguro financiado, nos bancos que publicam o CLTV com o seguro: na compra, o
  banco negou 10,8% dos cravados no teto com seguro e 9,8% do resto da faixa; no
  saque, 42,5% e 35,3%. Nos bancos que publicam sem o seguro, o resto da faixa foi
  negado em 91,6% no saque e 86,4% no sem saque, e em 16,0% na compra.
- Concordância com o banco: acima de 60% há 12.324 negados e 422 aprovados, e os
  refinanciamentos de 44% a 49% acima do teto somam 830, com 94,3% negados. Até
  43%, as duas subfaixas têm tamanho parecido, e 58.465 dos 94.533 pedidos têm CLTV
  acima de 90%.

**O que significa:**

- A parte comparada com a decisão do banco tem 360.096 pedidos, com negativas nas
  duas pontas de DTI e nos refinanciamentos acima do teto.
- Acima do teto do CLTV, o banco quase sempre negou, como a norma manda. No segundo
  financiamento a confirmar, aprovou quase todos, e a base não mostra o que permitiu
  a aprovação. Por isso, nesses pedidos, o gabarito é suspender pedindo a
  comprovação, e não a decisão do banco.
- Nos bancos que publicam com o seguro, a faixa do seguro cumpre a norma, e na compra
  os bancos agem assim. No saque, mesmo nesses bancos, cerca de 4 em 10 foram
  negados, e a base não mostra o motivo. Nos bancos que publicam sem o seguro, a
  faixa já conta como acima do teto (seção 4.6), e os refinanciamentos foram quase
  todos negados. O resultado da faixa sai separado na Evaluation, pelas colunas
  `seguro_financiado` e `cltv_publicado`.
- As duas subfaixas até 43% têm casos suficientes para serem lidas separadamente. Na
  maior parte dos pedidos até 43%, o CLTV passa de 90%, ponto a partir do qual a
  norma pede score de 580, que o HMDA não publica. A coluna `cltv_90` permite ver se
  o agente erra mais nesses pedidos.

<a id="secao-6"></a>

## 6. NSMO: os casos do segundo teste

**A pergunta.** Quais casos da NSMO testam o que o HMDA não consegue testar?

O HMDA não publica score nem reserva, e são eles que decidem a faixa de 44% a 50% de
DTI na norma. Nessa faixa, a norma pede score a partir de 580 e fator compensatório,
e a reserva de pelo menos três prestações é um dos fatores. A NSMO traz os dois, mas só de financiamentos
aprovados. Por isso o segundo teste usa a NSMO só nessa faixa.

A célula separa os financiamentos FHA (`loan_type` = 2), calcula o menor score entre
os tomadores (seção 6.2) e cruza a faixa de DTI com a presença do score do
respondente e da resposta sobre reserva (`x5001`).

In [45]:
fha = nsmo[nsmo['loan_type'] == LOAN_TYPE_FHA].copy()
SCORES = ['score_orig_r', 'score_orig_s', 'score_orig_o1', 'score_orig_o2',
          'score_orig_o3']
scores = fha[SCORES]
scores = scores.where(scores > 0)  # códigos negativos são ausência
fha['menor_score'] = scores.min(axis=1)
# O caso entra com o score do respondente e a reserva; o score usado na norma é o
# menor entre os tomadores (seção 6.2).
fha['tem_score_e_reserva'] = (fha['score_orig_r'] > 0) & (fha['x5001'] > 0)


def faixa_nsmo(dti):
    """Faixa de DTI do segundo teste."""
    if dti <= 43:
        return 'até 43'
    if dti <= 50:
        return '44 a 50'
    if dti <= 60:
        return '51 a 60'
    return 'acima de 60'


fha['faixa_dti'] = fha['dti'].map(faixa_nsmo)
print('FHA na NSMO:', br(len(fha)))
tabela = pd.crosstab(fha['faixa_dti'], fha['tem_score_e_reserva'].map(
    {True: 'com score e reserva', False: 'sem score ou reserva'}
), margins=True, margins_name='total')
tabela.reindex(FAIXAS[:1] + ['44 a 50', '51 a 60', 'acima de 60', 'total'])

FHA na NSMO: 7.331


tem_score_e_reserva,com score e reserva,sem score ou reserva,total
faixa_dti,,,
até 43,1243,2775,4018
44 a 50,718,1152,1870
51 a 60,620,699,1319
acima de 60,7,117,124
total,2588,4743,7331


**Onde olhar:** a linha "44 a 50", coluna "com score e reserva".

**O que se vê:** dos 7.331 financiamentos FHA, 1.870 têm DTI de 44% a 50%, e 718
deles têm score e resposta sobre reserva. As outras faixas ficam fora do teste. Até
43%, a norma não pede fator. De 51% a 60% (620 casos com score e reserva), a análise
manual não aprova, e esses casos entram só como contagem no relatório à parte.

**O que significa:** o segundo teste começa com 718 casos. A tabela abaixo mostra a
finalidade que eles declararam na pesquisa.

In [46]:
teste = fha[(fha['faixa_dti'] == '44 a 50') & fha['tem_score_e_reserva']].copy()
print('casos de 44% a 50% com score e reserva:', br(len(teste)))
pd.crosstab(teste['x33'].map(X33), teste['cashout'].map(
    {1: 'com saque', 2: 'sem saque'}
), margins=True, margins_name='total')

casos de 44% a 50% com score e reserva: 718


cashout,com saque,sem saque,total
x33,,,
compra,0,503,503
construção,0,1,1
imóvel quitado,0,5,5
inclusão ou retirada de co-tomador,5,2,7
refinanciamento,130,72,202
total,135,583,718


### 6.1 Finalidade

**Pergunta:** a finalidade da NSMO corresponde às três finalidades da tabela de
tetos?

A pergunta da pesquisa (`x33`, tabela acima) tem cinco respostas, e a marca de saque
(`cashout`) vem calculada pela NMDB. A célula traduz tudo para os códigos do HMDA,
com `nsmo_purpose_code` (`src/case_profile.py`), a mesma regra que monta o perfil do
caso. Compra vira compra. Refinanciamento e troca de co-tomador viram refinanciamento com
ou sem saque, conforme a marca, porque a troca de co-tomador refinancia a dívida que
já existe. Construção e imóvel quitado ficam como "outra", sem teto na tabela.

In [47]:
codigos = []
for x33, cashout in zip(teste['x33'], teste['cashout']):
    codigos.append(nsmo_purpose_code(x33, cashout))
teste['loan_purpose'] = codigos
teste['finalidade'] = teste['loan_purpose'].map(LOAN_PURPOSE).fillna('outra')
teste['finalidade'].value_counts()

finalidade
compra                       503
refinanciamento com saque    135
refinanciamento sem saque     74
outra                          6
Name: count, dtype: int64

**Onde olhar:** a contagem por finalidade.

**O que se vê:** 503 compras, 135 refinanciamentos com saque, 74 sem saque e 6 casos
"outra" (1 construção e 5 imóveis quitados).

**O que significa:** os 6 casos "outra" não têm teto na tabela e saem do teste na
seção 6.3, pelo mesmo motivo das finalidades sem teto do HMDA (seção 4.5).

**Pergunta:** a marca de saque da NSMO segue a definição do FHA?

A marca é calculada pela NMDB: "If the total value of the new loans was more than 5%
larger than that of the old loans, the new loan was labeled cash-out" (codebook da
NSMO, p. 151). Para o FHA, saque é o cliente levar mais de US$ 500 (Handbook, p.
467). Um refinanciamento que embute custos e o seguro inicial pode passar de 5% sem
dar nenhum dinheiro ao cliente. A tabela cruza o CLTV dos refinanciamentos com os dois
tetos: saque (80%, ou 81,4% com o seguro) e sem saque (97,75%, ou 99,5% com o
seguro).

In [48]:
# CLTV dos refinanciamentos da NSMO, contra os tetos do saque (80%, 81,4% com o
# seguro) e do sem saque (97,75%, 99,5% com o seguro).
refinanciamentos = teste[teste['loan_purpose'].isin(['31', '32'])]
cortes = [0, CLTV_CAP['32'], cap_with_ufmip('32', None), 90,
          CLTV_CAP['31'], cap_with_ufmip('31', None), 200]
faixas_cltv = pd.cut(refinanciamentos['cltv'], cortes)
pd.crosstab(faixas_cltv, refinanciamentos['finalidade'], margins=True,
            margins_name='total')

finalidade,refinanciamento com saque,refinanciamento sem saque,total
cltv,,,
"(0.0, 80.0]",69,23,92
"(80.0, 81.4]",47,2,49
"(81.4, 90.0]",8,9,17
"(90.0, 97.75]",8,26,34
"(97.75, 99.5]",3,14,17
total,135,74,209


**Onde olhar:** a coluna "refinanciamento com saque", nas linhas acima de 81,4.

**O que se vê:** 116 dos 135 refinanciamentos com saque estão até 81,4%, dentro do
teto do saque com o seguro. Os outros 19 vão de 81,4% a 99,5%, sendo 8 até 90%, 8 de 90% a
97,75% e 3 acima disso. Todos foram aprovados, porque a NSMO só tem aprovados.

Pela norma, um saque com CLTV de 93% não seria aprovado, mas num refinanciamento sem
saque esse valor cabe no teto. Há duas explicações possíveis para esses 19. Uma é a
marca de saque, que segue a regra da NMDB e não a do FHA. A outra é o ano, porque os
empréstimos da NSMO vão de 2019 a 2024 e o gabarito usa a regra de 2026.

**Pergunta:** as regras que o gabarito usa mudaram entre 2016 e 2026?

O gabarito segue a Update 18 do Handbook. A célula procura, em cada uma das nove
versões baixadas, a frase de onde sai cada número de `src/rules.py`. A tabela
manual e os fatores compensatórios já foram comparados no notebook 01 (seção 5.4) e
não mudaram.

In [49]:
# Frase do Handbook de onde sai cada número de src/rules.py.
REGRAS_HANDBOOK = {
    'score mínimo de 500':
        'not eligible for FHA-insured financing if the MDCS is less than 500',
    'score de 500 a 579, LTV até 90%':
        'between 500 and 579 limited to a maximum LTV of 90%',
    'reserva de 3 prestações':
        'Reserves are equal to or exceed three total monthly Mortgage Payments',
    'compra, LTV até 96,5%':
        'For purchase transactions, the maximum LTV is 96.5 percent',
    'sem saque, CLTV até 97,75%':
        'The maximum CLTV ratio for a Rate and Term refinance is 97.75 percent',
    'seguro inicial de 1,75%': '175 Basis Points (bps) (1.75%)',
    'segundo financiamento público sem teto de CLTV':
        'There is no maximum Combined Loan-to-Value (CLTV) for secondary financing'
        ' meeting the requirements found in Governmental Entities',
    'saque é levar mais de US$ 500': 'greater than $500 cash back to the Borrower',
    'saque, CLTV até 85%': 'The maximum CLTV is 85 percent of the Adjusted Value',
    'saque, CLTV até 80%': 'The maximum CLTV is 80 percent of the Adjusted Value',
}

linhas = []
for regra, frase in REGRAS_HANDBOOK.items():
    linha = {'regra': regra}
    for versao in HANDBOOK:
        if has_sentence(versao, frase):
            linha[versao] = 'sim'
        else:
            linha[versao] = 'não'
    linhas.append(linha)
pd.DataFrame(linhas).set_index('regra')

,2016_12,2019_03,2019_07,2019_08,2019_10,2019_12,2020_11,update_17,update_18
regra,,,,,,,,,
score mínimo de 500,sim,sim,sim,sim,sim,sim,sim,sim,sim
"score de 500 a 579, LTV até 90%",sim,sim,sim,sim,sim,sim,sim,sim,sim
reserva de 3 prestações,sim,sim,sim,sim,sim,sim,sim,sim,sim
"compra, LTV até 96,5%",sim,sim,sim,sim,sim,sim,sim,sim,sim
"sem saque, CLTV até 97,75%",sim,sim,sim,sim,sim,sim,sim,sim,sim
"seguro inicial de 1,75%",sim,sim,sim,sim,sim,sim,sim,sim,sim
segundo financiamento público sem teto de CLTV,sim,sim,sim,sim,sim,sim,sim,sim,sim
saque é levar mais de US$ 500,sim,sim,sim,sim,sim,sim,sim,sim,sim
"saque, CLTV até 85%",sim,sim,sim,sim,não,não,não,não,não


**Onde olhar:** as duas últimas linhas, do teto do saque.

**O que se vê:** as oito primeiras regras aparecem nas nove versões. O teto do saque
é de 85% até a versão de agosto de 2019 e de 80% a partir da de outubro de 2019, que
cita a Mortgagee Letter 2019-11 como origem da mudança.

**Pergunta:** os 19 saques acima do teto são de antes da mudança?

In [50]:
saques_acima = refinanciamentos[
    (refinanciamentos['loan_purpose'] == '32')
    & (refinanciamentos['cltv'] > cap_with_ufmip('32', None))
]
pd.crosstab(saques_acima['open_year'], saques_acima['cltv'], margins=True,
            margins_name='total')

cltv,85,86,87,88,89,91,92,93,95,97,98,99,total
open_year,,,,,,,,,,,,,
2019,1,3,0,0,0,1,0,0,0,0,0,1,6
2021,0,0,1,1,1,0,0,0,0,0,1,0,4
2022,0,0,0,0,0,0,0,1,0,0,0,0,1
2023,0,0,0,1,0,0,1,1,0,0,1,0,4
2024,0,0,0,0,0,0,0,1,2,1,0,0,4
total,1,3,1,2,1,1,1,3,2,1,2,1,19


**Onde olhar:** a linha de 2019.

**O que se vê:** 6 dos 19 são de 2019. Quatro têm CLTV de 85% ou 86%, que cabe no
teto antigo de 85% (86,5% com o seguro). Os outros dois, com 91% e 99%, e os 13 de
2021 a 2024 passam do teto em qualquer versão.

**O que significa:** a NSMO só informa o ano do empréstimo, então não se sabe se os 4
de 2019 vieram antes ou depois da mudança. Neles, a regra antiga explica o CLTV tão
bem quanto a marca de saque. Nos outros 15, sobra a marca. Nas duas explicações, o
gabarito estaria negando por garantia um empréstimo que o banco podia aprovar, e por
isso os casos acima de 81,4% saem do teste na seção 6.3 (15 deles; os outros 4 já
saem antes, por estarem fora da tabela manual). Até 81,4%, nada disso muda a
resposta, porque os dois tetos são cumpridos. A garantia continua testada no HMDA,
que tem negativas reais.

### 6.2 Score: o menor entre os tomadores

**Pergunta:** qual score usar quando há mais de um tomador?

O Handbook manda usar o menor score entre os tomadores (II.A.1.b.ii(A)(3)). A NSMO
traz o do respondente (`score_orig_r`), o do cônjuge (`score_orig_s`) e até três
outros (`score_orig_o1` a `o3`), e os códigos negativos são ausência. A célula aplica
a tabela manual com o menor score e conta os casos em cada leitura.

A tabela manual (no código, "matriz manual") tem uma linha por faixa de DTI e PTI, e
o score decide quais linhas valem. Com 580 ou mais, ela aceita DTI até 43% (PTI até
31%) sem fator compensatório, até 47% (PTI até 37%) com um e até 50% (PTI até 40%)
com dois. De 500 a 579, só vale a linha de até 43%, sem fator. Como a reserva é o
único fator que a NSMO mostra, cada caso recebe uma destas leituras:

- **exigência verificada:** o caso cabe numa linha e cumpre o que ela pede, seja
  porque a linha não pede fator, seja porque pede um e a reserva cobre;
- **fator não verificado:** a linha pede um fator (DTI até 47%) e a reserva não
  chega a 3 prestações;
- **1 de 2 fatores verificado:** a linha pede dois fatores (DTI até 50%) e a reserva
  cobre só um;
- **nenhum de 2 fatores verificado:** a linha pede dois fatores e a reserva não cobre
  nenhum;
- **fora da matriz manual:** o caso não cabe em nenhuma linha (DTI acima de 50%, PTI
  acima de 40%, score de 500 a 579 com DTI acima de 43%, ou score abaixo de 500). A
  análise manual não aprova esses casos.

In [51]:
teste['pti_num'] = teste['pti'].where(teste['pti'] > 0)
teste['tem_reserva'] = teste['x5001'].isin([3, 4])

leituras = []
for dti, pti, menor, reserva in zip(
    teste['dti'], teste['pti_num'], teste['menor_score'], teste['tem_reserva'],
):
    leituras.append(norm_reading(manual_requirement(dti, pti, menor), reserva))
teste['leitura'] = leituras

teste['leitura'].value_counts().rename_axis('leitura').rename('casos').to_frame()

,casos
leitura,
fator não verificado,192
nenhum de 2 fatores verificado,154
fora da matriz manual,154
1 de 2 fatores verificado,120
exigência verificada,98


**O que se vê:** dos 718 casos, 98 têm a exigência verificada, 192 têm o fator não
verificado, 120 têm 1 de 2 fatores, 154 não têm nenhum dos 2 e 154 ficam fora da
matriz manual.

**O que significa:** o teste usa o menor score, como manda a norma, e é esse score que
o perfil do agente mostra. A diferença entre o VantageScore 3.0 da NSMO e o score
usado pelo banco fica como limitação.

### 6.3 O que a norma permite concluir em cada caso

**Pergunta:** com DTI, PTI, menor score, reserva e CLTV, o que a norma em código
responde em cada caso?

A célula chama `expected_nsmo` (`src/rules.py`) e cruza a leitura da tabela manual com
a situação do CLTV. A reserva conta como fator verificado quando o cliente declarou 3
meses de prestação ou mais (II.A.5.d.ix pede três prestações para imóvel de 1 ou 2
unidades). É o único fator compensatório que a NSMO mostra. A segunda célula diz em
que parte da avaliação cada caso entra.

Antes, vale a mesma pergunta da seção 4.6, se a NSMO publica o CLTV com o seguro
somado. Ela
não identifica o banco, então a contagem é feita na base inteira. O CLTV vem em
número inteiro, e por isso o teto da compra (96,5) cai em 96 ou 97, o teto com o
seguro (98,19) em 98, o do saque (80) em 80, o do saque com o seguro (81,4) em 81, o
do sem saque (97,75) em 98 e o do sem saque com o seguro (99,46) em 99.

In [52]:
for finalidade in ['compra', 'refinanciamento com saque',
                   'refinanciamento sem saque']:
    da_finalidade = teste[teste['finalidade'] == finalidade]
    perto = da_finalidade[(da_finalidade['cltv'] >= 78)
                          & (da_finalidade['cltv'] <= 100)]
    contagem = perto['cltv'].value_counts().sort_index()
    contagem = contagem.rename_axis('CLTV').rename('casos')
    print('NSMO,', finalidade, '(CLTV de 78 a 100)')
    display(contagem.to_frame().T)

NSMO, compra (CLTV de 78 a 100)


CLTV,80,81,82,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100
casos,2,4,2,3,4,2,1,3,2,4,5,18,9,19,25,68,77,195,3,3


NSMO, refinanciamento com saque (CLTV de 78 a 100)


CLTV,78,79,80,81,85,86,87,88,89,91,92,93,95,97,98,99
casos,6,2,3,47,1,3,1,2,1,1,1,3,2,1,2,1


NSMO, refinanciamento sem saque (CLTV de 78 a 100)


CLTV,79,80,81,83,85,87,88,89,93,94,95,96,97,98,99
casos,5,2,2,1,1,4,2,1,2,2,7,9,6,11,3


**Onde olhar:** o valor com mais casos em cada finalidade.

**O que se vê:** na compra, o pico está em 98, com 195 casos, contra 77 em 97 e 68 em
96. No saque, está em 81, com 47 casos, contra 3 em 80. No sem saque, o maior número
está em 98, que é o teto sem o seguro, com 11 casos, contra 9 em 96, 6 em 97 e 3 em
99, onde cairia o teto com o seguro.

**O que significa:** na compra e no saque, a NSMO publica o CLTV com o seguro somado,
o que faz sentido, porque o valor vem do registro do empréstimo, que já inclui o
seguro financiado. No sem saque, o dado não responde. Os 11 casos em 98 apontam para
o teto sem o seguro, mas são poucos, e nessa finalidade nem se espera pico, porque a
dívida nova vem da antiga e o cliente não escolhe o valor. Fica a premissa de que o
sem saque segue a mesma regra das outras finalidades, já que o CLTV de todos os
financiamentos sai da mesma fonte e do mesmo cálculo. Todos os casos da NSMO entram
como "com o seguro", e a faixa entre o teto e o teto com o seguro cumpre a norma. No
sem saque, a premissa decide a resposta dos 14 casos em 98 e 99, que a Evaluation
pode separar pela coluna `situacao_cltv` (tabela de limitações).

In [53]:
situacoes = []
respostas = []
for linha in teste.itertuples():
    situacoes.append(cltv_situation(linha.loan_purpose, linha.cltv,
                                    linha.menor_score, True))
    respostas.append(expected_nsmo(
        linha.dti, linha.pti_num, linha.menor_score, linha.tem_reserva,
        linha.loan_purpose, linha.cltv, True,
    ))
teste['situacao_cltv'] = situacoes
acoes, pendencias, motivos_negativa = split_answers(respostas)
teste['resposta_esperada'] = acoes
teste['pendencias'] = pendencias
teste['motivos'] = motivos_negativa
pd.crosstab(teste['leitura'], teste['situacao_cltv'], margins=True,
            margins_name='total')

situacao_cltv,acima do teto,dentro do teto,segundo financiamento a confirmar,seguro financiado,sem teto na tabela,total
leitura,,,,,,
1 de 2 fatores verificado,3,55,3,58,1,120
exigência verificada,3,50,5,40,0,98
fator não verificado,3,79,6,103,1,192
fora da matriz manual,7,70,19,56,2,154
nenhum de 2 fatores verificado,6,67,12,67,2,154
total,22,321,45,324,6,718


In [54]:
def onde_entra_nsmo(linha):
    """Em que parte da avaliação o caso da NSMO é medido."""
    if linha['finalidade'] == 'outra':
        return 'fora do teste (finalidade sem teto na tabela)'
    if linha['leitura'] == 'fora da matriz manual':
        if linha['menor_score'] < 500:
            return 'fora da tabela manual (score abaixo de 500)'
        if linha['menor_score'] < 580:
            return 'fora da tabela manual (score abaixo de 580)'
        return 'fora da tabela manual (PTI acima de 40%)'
    # Acima do teto do saque com o seguro, o caso pode ser um sem saque com a marca
    # da NMDB (dívida nova 5% maior que a antiga) ou um saque de 2019 sob o teto
    # antigo de 85% (seção 6.1). Nos dois, o gabarito negaria por garantia sem razão.
    if linha['loan_purpose'] == '32' and linha['situacao_cltv'] == 'acima do teto':
        return 'fora do teste (saque acima do teto)'
    if linha['resposta_esperada'] == DENY:
        return 'contra a norma (garantia acima do teto)'
    if linha['pendencias'] == '':
        return 'checagem (exigência verificada)'
    if 'segundo financiamento' in linha['pendencias']:
        return 'contra a norma (segundo financiamento)'
    return 'contra a norma (fator compensatório)'


destinos = []
for _, linha in teste.iterrows():
    destinos.append(onde_entra_nsmo(linha))
teste['onde_entra'] = destinos
pd.crosstab(teste['onde_entra'], teste['resposta_esperada'], margins=True,
            margins_name='total')

resposta_esperada,aprovar com fator compensatório,negar,suspender,total
onde_entra,,,,
checagem (exigência verificada),90,0,0,90
contra a norma (fator compensatório),0,0,429,429
contra a norma (segundo financiamento),0,0,26,26
fora da tabela manual (PTI acima de 40%),0,106,0,106
fora da tabela manual (score abaixo de 500),0,1,0,1
fora da tabela manual (score abaixo de 580),0,45,0,45
fora do teste (finalidade sem teto na tabela),0,2,4,6
fora do teste (saque acima do teto),0,15,0,15
total,90,169,459,718


**Onde olhar:** na primeira tabela, a linha "total"; na segunda, linha a linha.

**O que se vê:** na primeira tabela, 321 casos estão dentro do teto, 324 na faixa do
seguro, 45 compras acima dela (segundo financiamento a confirmar), 22 refinanciamentos
acima do teto e 6 sem teto na tabela. Na segunda:

- 90 casos de **checagem**: a exigência da tabela manual está verificada na base, e a
  resposta esperada é aprovar com fator compensatório.
- 429 **contra a norma (fator compensatório)**: falta um fator que a base não mostra,
  e a resposta esperada é suspender pedindo a comprovação.
- 26 **contra a norma (segundo financiamento)**: compra com CLTV acima de 98,2%, e a
  resposta esperada é suspender pedindo a comprovação.
- 152 **fora da tabela manual**: PTI acima de 40% (106), score abaixo de 580 (45) ou
  abaixo de 500 (1). A análise manual não aprova esses casos, então a aprovação veio
  de outro caminho, provavelmente o sistema automático.
- 21 **fora do teste**: 15 por saque acima do teto (seção 6.1) e 6 por finalidade sem
  teto.

**O que significa:** o segundo teste fica com 545 casos, 90 de checagem e 455 contra
a norma. Os 152 fora da tabela manual e os 21 fora do teste não entram nos grupos.

### 6.4 Sorteio dos grupos na NSMO

**Pergunta:** como dividir os 545 casos entre ajuste e nota final?

Os casos contra a norma são sorteados como no HMDA, com 30% para o ajuste dentro de
cada finalidade e a mesma semente. Os 90 de checagem têm outra divisão. São os únicos
casos do projeto em que a resposta certa é aprovar com fator compensatório, porque o
HMDA não mostra score nem reserva. Se ficassem todos na nota final, o prompt nunca
veria essa resposta no ajuste, e um agente que suspende toda a faixa de 44% a 50%
pareceria certo até a rodada final. Por isso 10% deles vão para o ajuste, o bastante
para ver a resposta, e o resto fica na nota final, que precisa de casos. A célula
sorteia a checagem depois das outras partes, para não mudar o sorteio delas, e
imprime a margem de erro com 80% de acerto.

In [55]:
SHARE_CHECK_TUNING = 0.10  # parte da checagem que vai para o ajuste


def margem(n, acerto=0.8):
    """Margem de erro de 95%, em pontos, de uma taxa de acerto medida em n casos."""
    return 1.96 * (acerto * (1 - acerto) / n) ** 0.5 * 100


teste['grupo'] = ''
sorteaveis = teste['onde_entra'].str.startswith('contra a norma')
rng_nsmo = np.random.default_rng(SEED)
for finalidade in sorted(teste.loc[sorteaveis, 'finalidade'].unique()):
    indices = teste.index[sorteaveis & (teste['finalidade'] == finalidade)].to_numpy()
    embaralhados = rng_nsmo.permutation(indices)
    n_ajuste = round(len(indices) * SHARE_TUNING)
    teste.loc[embaralhados, 'grupo'] = 'nota final'
    teste.loc[embaralhados[:n_ajuste], 'grupo'] = 'ajuste'

checagem = teste.index[teste['onde_entra'] == 'checagem (exigência verificada)']
embaralhados = rng_nsmo.permutation(checagem.to_numpy())
n_ajuste = round(len(checagem) * SHARE_CHECK_TUNING)
teste.loc[embaralhados, 'grupo'] = 'nota final'
teste.loc[embaralhados[:n_ajuste], 'grupo'] = 'ajuste'

com_grupo = teste[teste['grupo'] != '']
final_nsmo = com_grupo[com_grupo['grupo'] == 'nota final']
n_checagem = int((final_nsmo['onde_entra'] == 'checagem (exigência verificada)').sum())
n_contra = int(final_nsmo['onde_entra'].str.startswith('contra a norma').sum())
print('margem, checagem na nota final:', n_checagem, 'casos, ±',
      round(margem(n_checagem), 1), 'pontos')
print('margem, checagem inteira na nota final:', len(checagem), 'casos, ±',
      round(margem(len(checagem)), 1), 'pontos')
print('margem, contra a norma na nota final:', n_contra, 'casos, ±',
      round(margem(n_contra), 1), 'pontos')
pd.crosstab(com_grupo['onde_entra'], com_grupo['grupo'], margins=True,
            margins_name='total')

margem, checagem na nota final: 81 casos, ± 8.7 pontos
margem, checagem inteira na nota final: 90 casos, ± 8.3 pontos
margem, contra a norma na nota final: 319 casos, ± 4.4 pontos


grupo,ajuste,nota final,total
onde_entra,,,
checagem (exigência verificada),9,81,90
contra a norma (fator compensatório),127,302,429
contra a norma (segundo financiamento),9,17,26
total,145,400,545


**Onde olhar:** a linha "total" e as margens impressas.

**O que se vê:** 145 casos no ajuste e 400 na nota final. Da checagem, 9 vão para o
ajuste e 81 ficam na nota final, com margem de erro de cerca de 8,7 pontos, contra
8,3 se ficassem os 90. Os 319 casos contra a norma da nota final (302 do fator
compensatório e 17 do segundo financiamento) dão margem de 4,4 pontos.

**O que significa:** o ajuste do prompt na NSMO passa a ter casos com as duas
respostas da faixa de 44% a 50%, suspender pedindo o fator e aprovar com ele. A
margem da checagem na nota final quase não muda.

<a id="secao-7"></a>

## 7. Resposta esperada pela norma em código

**A pergunta.** O que a norma, aplicada ao pé da letra em código, responde em cada
pedido do HMDA?

A função `expected_hmda` (`src/rules.py`) aplica a tabela manual e os tetos de CLTV
ao DTI publicado, à finalidade e ao CLTV:

- DTI até 43%: aprovar;
- 44% a 47%: suspender pedindo 1 fator compensatório; 48% a 49%: suspender pedindo 2;
- 50% a 60% e acima de 60%: negar por DTI;
- refinanciamento acima do teto: negar por garantia;
- compra acima do teto: suspender pedindo a comprovação do segundo financiamento.

O teto conta com o seguro somado só nos bancos que publicam o CLTV com ele (98,2% na
compra, 81,4% no saque e 99,5% no sem saque); nos demais, vale o teto da norma
(96,5%, 80% e 97,75%), como decidido na seção 4.6.

Quando há mais de uma conclusão, vale o custo assimétrico do projeto, e por isso
negar vence suspender, e suspender vence aprovar. Score, PTI e reserva ausentes são lacuna da base
e não viram pendência do cliente. A resposta fica numa coluna fixa, gravada antes de
existir o agente, e serve de teto de referência na Evaluation.

A primeira célula roda os testes de `tests/test_rules.py`, com um caso por linha da
tabela manual, por situação de CLTV e por tipo de resposta. A segunda calcula a
resposta em todos os pedidos, porque ela vai gravada no arquivo de saída, mas só a
compara com a decisão do banco no grupo de ajuste. A nota final continua fechada até
a Evaluation.

In [56]:
resultado = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests', '-q'],
    cwd=ROOT, capture_output=True, text=True,
)
print(resultado.stdout.strip().splitlines()[-1])

39 passed in 0.21s


In [57]:
respostas = []
for dti, finalidade, cltv, publicado in zip(
    casos['debt_to_income_ratio'], casos['loan_purpose'], casos['cltv'],
    casos['cltv_publicado'],
):
    com_seguro = publicado == 'com o seguro'
    respostas.append(expected_hmda(dti, finalidade, cltv, com_seguro))
acoes, pendencias, motivos_negativa = split_answers(respostas)
casos['resposta_esperada'] = acoes
casos['pendencias'] = pendencias
casos['motivos'] = motivos_negativa

ajuste = casos[casos['grupo'] == 'ajuste']
pd.crosstab([ajuste['onde_entra'], ajuste['resposta_esperada']], ajuste['decisao'],
            rownames=['parte da avaliação', 'resposta da norma'],
            colnames=['decisão do banco'], margins=True, margins_name='total')

decisão do banco                                          aprovado  negado   total
parte da avaliação                     resposta da norma                          
concordância com o banco               aprovar               87838    5079   92917
                                       negar                   583   14609   15192
contra a norma (fator compensatório)   suspender             66850    4004   70854
contra a norma (segundo financiamento) suspender             16690     629   17319
relatório à parte (DTI de 50% a 60%)   negar                 74941   10666   85607
total                                                       246902   34987  281889

**Onde olhar:** a primeira linha da saída e, na tabela, o cruzamento entre a resposta
da norma (linhas) e a decisão do banco (colunas).

**O que se vê:** os 36 testes passam. No grupo de ajuste, na concordância com o banco,
a norma aprova 92.917 pedidos (87.838 aprovados pelo banco e 5.079 negados) e nega
15.192 (14.609 negados e 583 aprovados). Nas partes contra a norma, a resposta é
sempre suspender (70.854 e 17.319 pedidos). No relatório à parte, a resposta é
sempre negar, e 74.941 desses pedidos foram aprovados pelo banco.

**O que significa:** onde a norma dá resposta direta, ela coincide com o banco na
grande maioria dos pedidos. A maior discordância está nos 5.079 negados que a norma
aprovaria, negativas cujo motivo declarado o DTI e o CLTV publicados não explicam. No
relatório à parte, a discordância é esperada, porque as aprovações de 50% a 60% vêm do
sistema automático.

### 7.1 Referências e teto, com e sem os removidos

**Pergunta:** as remoções da seção 3 favorecem alguma das respostas?

Tirar negativas sem motivo visível sobe o acerto de qualquer regra, porque nenhuma
acerta uma negativa que o dado não explica. O que importa é se sobe mais para uma do
que para outra. A tabela compara, na concordância com o banco e só no grupo de
ajuste, três respostas. São elas sempre aprovar, a regra fixa de DTI (até 43% aprova,
acima nega) e a norma em código. A conta é feita duas vezes, sem os removidos e com as negativas da seção 3 de volta
(as que têm DTI). O grupo da nota final não é tocado.

In [58]:
def regra_fixa(faixa):
    """Adversário simples: DTI até 43% aprova, acima nega."""
    if faixa == 'até 43':
        return APPROVE
    return DENY


def acerto(acoes, decisoes):
    """Percentual de pedidos em que a ação bate com a decisão do banco."""
    bate = ((acoes == APPROVE) & (decisoes == 'aprovado')) | (
        (acoes == DENY) & (decisoes == 'negado')
    )
    return round(100 * bate.mean(), 1)


# "Com os removidos": as negativas que saíram nas decisões da seção 3 voltam,
# desde que tragam DTI (as referências precisam dele).
SECAO_3 = [
    'negado por cadastro incompleto',
    'motivo invisível sem motivo visível',
    'grupo outros sem motivo visível',
    'motivo visível sem o campo publicado',
]
removidos = hmda[hmda['sai_por'].isin(SECAO_3) & hmda['tem_dti']].copy()
situacoes = []
respostas = []
for dti, finalidade, cltv, publicado in zip(
    removidos['debt_to_income_ratio'], removidos['loan_purpose'], removidos['cltv'],
    removidos['cltv_publicado'],
):
    com_seguro = publicado == 'com o seguro'
    situacao = cltv_situation(finalidade, cltv, None, com_seguro)
    situacoes.append(situacao)
    respostas.append(expected_hmda(dti, finalidade, cltv, com_seguro)['acao'])
removidos['situacao_cltv'] = situacoes
removidos['resposta_esperada'] = respostas
destinos = []
for faixa, situacao in zip(removidos['faixa_dti'], removidos['situacao_cltv']):
    destinos.append(onde_entra(faixa, situacao))
removidos['onde_entra'] = destinos

colunas = ['faixa_dti', 'decisao', 'resposta_esperada', 'onde_entra', 'grupo']
sem = casos[colunas]
com = pd.concat([sem, removidos[colunas]])

linhas = []
for nome, tabela in [('sem os removidos', sem), ('com os removidos', com)]:
    medida = tabela[(tabela['grupo'] == 'ajuste')
                    & (tabela['onde_entra'] == 'concordância com o banco')]
    sempre = pd.Series(APPROVE, index=medida.index)
    fixa = medida['faixa_dti'].map(regra_fixa)
    linhas.append({
        'base': nome,
        'pedidos': len(medida),
        '% negados': round(100 * (medida['decisao'] == 'negado').mean(), 1),
        'sempre aprovar (%)': acerto(sempre, medida['decisao']),
        'DTI até 43% aprova (%)': acerto(fixa, medida['decisao']),
        'norma em código (%)': acerto(medida['resposta_esperada'], medida['decisao']),
    })
pd.DataFrame(linhas).set_index('base')

,pedidos,% negados,sempre aprovar (%),DTI até 43% aprova (%),norma em código (%)
base,,,,,
sem os removidos,108109,18.2,81.8,93.5,94.8
com os removidos,125206,29.4,70.6,82.8,84.1


**Onde olhar:** as três colunas de acerto, nas duas linhas.

**O que se vê:** sem os removidos, são 108.109 pedidos, com 18,2% de negados. Sempre
aprovar acerta 81,8%, a regra fixa 93,5% e a norma em código 94,8%. Com os removidos,
são 125.206 pedidos e 29,4% de negados, e os acertos caem para 70,6%, 82,8% e 84,1%.

**O que significa:** as remoções não favorecem nenhuma das três respostas. Com os
removidos de volta, as três perdem quase o mesmo (11,2, 10,7 e 10,7 pontos), e a
ordem não muda. Se os removidos ficassem numa faixa em que uma regra acerta e a outra
erra, a ordem poderia se inverter, e a decisão de remover escolheria o vencedor. A
norma em código fica pouco acima da regra fixa porque, nas pontas de DTI, as duas quase sempre respondem igual, e a diferença
vem da garantia. Na Evaluation, o agente é comparado com as três, com os resultados
com e sem os removidos lado a lado.

<a id="secao-8"></a>

## 8. O que o agente vê

**A pergunta.** Que informação do pedido chega ao agente?

O agente recebe um texto com 12 campos, montado por `src/case_profile.py`. HMDA e
NSMO saem com os mesmos campos, na mesma ordem e no mesmo formato, e o campo que a
base não traz aparece como "não informado". Ficam de fora:

- a decisão do banco e os motivos da negativa, que são o gabarito;
- o banco (`lei`) e o nome da base;
- o código do sistema automático (`aus-1`), pelo motivo da seção 4.6;
- na NSMO, as perguntas sobre ajuda na entrada (`x36c` e `x36d`), que misturam doação
  com empréstimo e não confirmam o segundo financiamento que a norma aceita;
- as marcações da seção 5, como `seguro_financiado`, que são leitura do projeto sobre
  o dado. Entregá-las seria fazer o raciocínio pelo agente.

A exceção é o jeito como o CLTV foi publicado (`cltv_publicado`, seção 4.6), que
aparece entre parênteses na linha do CLTV. É um fato que o analista conhece pelo
contrato, e não uma leitura do projeto. Sem ele, o agente não teria como saber se um
CLTV de 98,5% no refinanciamento sem saque passou do teto. Na NSMO, a linha sempre
diz "com o seguro".

O nome da base fica de fora, mas os campos preenchidos e o formato do CLTV mostram de
onde o caso vem (seção 8.1).

### 8.1 Perfil do caso

**Pergunta:** como fica o texto de um caso real?

A célula sorteia, no grupo de ajuste, uma compra do HMDA na faixa do fator
compensatório e um caso da NSMO.

In [59]:
exemplo_hmda = casos[(casos['grupo'] == 'ajuste')
                     & (casos['onde_entra'] == 'contra a norma (fator compensatório)')
                     & (casos['finalidade'] == 'compra')].sample(1, random_state=SEED)
exemplo_nsmo = teste[(teste['grupo'] == 'ajuste')
                     & teste['onde_entra'].str.startswith('contra a norma')
                     ].sample(1, random_state=SEED)

print('Um pedido do HMDA')
linha = exemplo_hmda.iloc[0]
com_seguro = linha['cltv_publicado'] == 'com o seguro'
print(case_profile(hmda_case(linha, com_seguro)))
# A resposta esperada não entra no perfil; aparece aqui só para o leitor.
print('-> norma em código:', linha['resposta_esperada'], '|', linha['pendencias'])
print()
print('Um caso da NSMO')
linha = exemplo_nsmo.iloc[0]
print(case_profile(nsmo_case(linha, linha['menor_score'])))
print('-> norma em código:', linha['resposta_esperada'], '|', linha['pendencias'])

Um pedido do HMDA
Finalidade: compra
Valor do empréstimo: US$ 375.000
Valor do imóvel: US$ 385.000
Renda anual: US$ 91.000
DTI: 49%
PTI: não informado
CLTV: 99,13% (sem o seguro inicial do FHA)
Score de crédito: não informado
Reserva após o fechamento: não informado
Ocupação: residência principal
Posição da hipoteca: primeira hipoteca
Unidades: 1
-> norma em código: suspender | comprovação de 2 fatores compensatórios; comprovação do segundo financiamento (II.A.5)

Um caso da NSMO
Finalidade: compra
Valor do empréstimo: não informado
Valor do imóvel: não informado
Renda anual: não informado
DTI: 46%
PTI: 25%
CLTV: 98% (com o seguro inicial do FHA, quando financiado)
Score de crédito: 651
Reserva após o fechamento: 1 a 2 meses de prestação
Ocupação: não informado
Posição da hipoteca: não informado
Unidades: não informado
-> norma em código: suspender | comprovação de 1 fator compensatório


**Onde olhar:** os campos "não informado" de cada perfil e a linha "norma em código"
abaixo dele.

**O que se vê:** o pedido do HMDA traz valores, renda, DTI de 49% e CLTV de 99,13%
publicado sem o seguro, mas não PTI, score nem reserva. O caso da NSMO traz DTI de
46%, PTI de 25%, CLTV de 98% com o seguro, score de 651 e reserva de 1 a 2 meses, mas não valores, renda, ocupação, posição
da hipoteca nem unidades.

**O que significa:** os dois perfis têm os mesmos campos, mas os preenchidos mudam com
a base, e o CLTV da NSMO vem sempre em número inteiro e "com o seguro". Quem lê o
perfil consegue deduzir de onde o caso vem. Isso não entrega a resposta, porque o
agente lê cada caso sozinho e só saberia que a NSMO tem apenas aprovados se o prompt
dissesse. Além disso, na NSMO o gabarito é a norma, que manda suspender em 455 dos
545 casos do teste. Para o pedido do HMDA, a norma em código responde suspender,
pedindo 2 fatores compensatórios e a comprovação do segundo financiamento. Para o
caso da NSMO, a reserva de 1 a 2 meses não chega às três prestações, e a resposta é
suspender pedindo 1 fator.

<a id="secao-9"></a>

## 9. Contagens para a Evaluation e conjuntos fixos

**A pergunta.** Onde a Evaluation precisa sortear mais casos, e que casos ficam
separados desde já?

O agente não roda nos 657 mil pedidos da nota final. A Evaluation vai sortear uma
amostra de cada parte da avaliação e, para isso, precisa saber quantos pedidos existem
em cada combinação de parte, faixa de DTI, finalidade e decisão. A tabela mostra essas
contagens no grupo de ajuste. Os dois grupos saíram do mesmo sorteio, então a nota
final tem perto da mesma proporção em cada combinação, com cerca de 2,3 vezes mais
pedidos (657.014 contra 281.889). A decisão do banco dentro da nota final só é
contada na Evaluation, na hora de sortear a amostra.

In [60]:
ajuste = casos[casos['grupo'] == 'ajuste']
celulas = pd.crosstab([ajuste['onde_entra'], ajuste['faixa_dti'],
                       ajuste['finalidade']],
                      ajuste['decisao'])
celulas

decisao                                                                       aprovado  negado
onde_entra                             faixa_dti   finalidade                                 
concordância com o banco               44 a 49     refinanciamento com saque        28     656
                                                   refinanciamento sem saque        19     127
                                       acima de 60 compra                          161    7006
                                                   refinanciamento com saque       225    4224
                                                   refinanciamento sem saque        36    1094
                                       até 43      compra                        62090    1185
                                                   refinanciamento com saque     20174    4563
                                                   refinanciamento sem saque      5688     833
contra a norma (fator compensatório)   44 a 49     compra                        53854    1809
                                                   refinanciamento com saque     10454    1830
                                                   refinanciamento sem saque      2542     365
contra a norma (segundo financiamento) até 43      compra                        16690     629
relatório à parte (DTI de 50% a 60%)   50 a 60     compra                        60182    4831
                                                   refinanciamento com saque     11925    4720
                                                   refinanciamento sem saque      2834    1115

**Onde olhar:** as combinações com menos pedidos em cada coluna.

**O que se vê:** algumas combinações são pequenas. A menor é a dos refinanciamentos
sem saque aprovados de 44% a 49% com CLTV acima do teto (19), que entram na
concordância com o banco. Na concordância acima de 60%, há 36 aprovados no sem saque
e 161 na compra; até 43%, 833 negados no sem saque. A maior é a compra aprovada até
43% (62.090).

**O que significa:** um sorteio simples traria quase só compras aprovadas até 43%. A
Evaluation fixa o número de casos por combinação antes de rodar o grupo final. Se a
proporção se repete, a menor combinação fica com cerca de 45 pedidos na nota final,
e o número de casos por combinação precisa caber nesse limite.

**Pergunta:** que casos do grupo de ajuste ficam separados para uso fixo?

São três conjuntos, sorteados com a semente do projeto:

- **regressão de prompt**: 15 pedidos do HMDA e 6 casos da NSMO, cobrindo as
  respostas típicas (aprovar até 43%, negar acima de 60%, negar saque por garantia,
  suspender pedindo fatores ou o segundo financiamento, aprovar com fator
  compensatório). Dois deles são
  refinanciamentos na faixa do seguro, um em banco que publica o CLTV sem o seguro
  (negar por garantia) e outro em banco que publica com ele (aprovar). O par mostra se
  o agente lê o jeito do banco na linha do CLTV. A cada mudança no prompt, esses casos
  rodam de novo, para ver se algo que funcionava quebrou;
- **demo**: 3 pedidos com renda zero ou negativa, removidos na seção 4.1, para mostrar
  o agente diante de um dado impossível;
- **checagem à parte**: 20 negados por cadastro incompleto sem DTI ou CLTV (seção
  3.1), para ver se o agente suspende pedindo o dado ou inventa um número.

In [61]:
ajuste = casos[casos['grupo'] == 'ajuste']
ajuste_nsmo = teste[teste['grupo'] == 'ajuste']

# Células da regressão de prompt: (descrição, máscara, quantos casos).
regressao = [
    ('HMDA até 43%, aprovado, a norma aprova',
     (ajuste['onde_entra'] == 'concordância com o banco')
     & (ajuste['faixa_dti'] == 'até 43') & (ajuste['decisao'] == 'aprovado')
     & (ajuste['resposta_esperada'] == APPROVE), 3),
    ('HMDA acima de 60%, negado, a norma nega por DTI',
     (ajuste['faixa_dti'] == 'acima de 60') & (ajuste['decisao'] == 'negado'), 3),
    ('HMDA saque acima de 80%, negado por garantia',
     (ajuste['loan_purpose'] == '32') & (ajuste['situacao_cltv'] == 'acima do teto')
     & (ajuste['decisao'] == 'negado') & ajuste['motivo_gar_visivel']
     & (ajuste['faixa_dti'] == 'até 43'), 2),
    ('HMDA 44% a 47%, suspender pedindo 1 fator',
     ajuste['pendencias'] == 'comprovação de 1 fator compensatório', 2),
    ('HMDA 48% a 49%, suspender pedindo 2 fatores',
     ajuste['pendencias'] == 'comprovação de 2 fatores compensatórios', 1),
    ('HMDA compra acima do teto, suspender pedindo o segundo financiamento',
     (ajuste['onde_entra'] == 'contra a norma (segundo financiamento)')
     & (ajuste['faixa_dti'] == 'até 43'), 2),
    ('HMDA faixa do seguro, banco sem o seguro, negar',
     ajuste['loan_purpose'].isin(['31', '32']) & (ajuste['seguro_financiado'] != '')
     & (ajuste['cltv_publicado'] == 'sem o seguro')
     & (ajuste['decisao'] == 'negado') & ajuste['motivo_gar_visivel']
     & (ajuste['faixa_dti'] == 'até 43'), 1),
    ('HMDA faixa do seguro, banco com o seguro, aprovar',
     ajuste['loan_purpose'].isin(['31', '32']) & (ajuste['seguro_financiado'] != '')
     & (ajuste['cltv_publicado'] == 'com o seguro')
     & (ajuste['decisao'] == 'aprovado') & (ajuste['resposta_esperada'] == APPROVE)
     & (ajuste['faixa_dti'] == 'até 43'), 1),
]
escolhidos = []
for celula, mascara, n in regressao:
    amostra = ajuste[mascara].sample(n, random_state=SEED)
    for indice in amostra.index:
        escolhidos.append({'base': 'HMDA', 'indice': indice,
                           'conjunto': 'regressão de prompt', 'celula': celula})
fator_nsmo = ajuste_nsmo[ajuste_nsmo['onde_entra']
                         == 'contra a norma (fator compensatório)']
for indice in fator_nsmo.sample(3, random_state=SEED).index:
    escolhidos.append({'base': 'NSMO', 'indice': indice,
                       'conjunto': 'regressão de prompt',
                       'celula': 'NSMO, fator não verificado, suspender'})
checagem_nsmo = ajuste_nsmo[ajuste_nsmo['onde_entra']
                            == 'checagem (exigência verificada)']
for indice in checagem_nsmo.sample(3, random_state=SEED).index:
    escolhidos.append({'base': 'NSMO', 'indice': indice,
                       'conjunto': 'regressão de prompt',
                       'celula': 'NSMO, exigência verificada, aprovar com fator'})

renda_zero = hmda[(hmda['sai_por'] == 'renda zero ou negativa')
                  & (hmda['grupo'] == 'ajuste')]
for indice in renda_zero.sample(3, random_state=SEED).index:
    escolhidos.append({'base': 'HMDA', 'indice': indice, 'conjunto': 'demo',
                       'celula': 'renda zero ou negativa'})

incompleto_vazio = hmda[(hmda['sai_por'] == 'negado por cadastro incompleto')
                        & (hmda['grupo'] == 'ajuste')
                        & (~hmda['tem_dti'] | ~hmda['tem_cltv'])]
for indice in incompleto_vazio.sample(20, random_state=SEED).index:
    escolhidos.append({'base': 'HMDA', 'indice': indice,
                       'conjunto': 'checagem à parte',
                       'celula': 'cadastro incompleto sem DTI ou CLTV'})

conjuntos = pd.DataFrame(escolhidos)
pd.crosstab([conjuntos['conjunto'], conjuntos['celula']], conjuntos['base'])

base                                                                    HMDA  NSMO
conjunto            celula                                                        
checagem à parte    cadastro incompleto sem DTI ou CLTV                   20     0
demo                renda zero ou negativa                                 3     0
regressão de prompt HMDA 44% a 47%, suspender pedindo 1 fator              2     0
                    HMDA 48% a 49%, suspender pedindo 2 fatores            1     0
                    HMDA acima de 60%, negado, a norma nega por DTI        3     0
                    HMDA até 43%, aprovado, a norma aprova                 3     0
                    HMDA compra acima do teto, suspender pedindo o ...     2     0
                    HMDA faixa do seguro, banco com o seguro, aprovar      1     0
                    HMDA faixa do seguro, banco sem o seguro, negar        1     0
                    HMDA saque acima de 80%, negado por garantia           2     0
                    NSMO, exigência verificada, aprovar com fator          0     3
                    NSMO, fator não verificado, suspender                  0     3

**Onde olhar:** a coluna de cada base.

**O que se vê:** são 44 casos, 21 na regressão de prompt (15 do HMDA e 6 da NSMO), 3
na demo e 20 na checagem à parte, todos do grupo de ajuste.

**O que significa:** nenhum desses casos entra na nota final. Os conjuntos ficam
gravados em arquivo (seção 10) e saem iguais em qualquer execução.

<a id="secao-10"></a>

## 10. Saídas

**A pergunta.** O que este notebook entrega para as fases seguintes?

Três arquivos em `data/processed/`:

- `hmda_casos.parquet`: os 938.903 pedidos do HMDA que ficam, com as colunas
  originais, o grupo, as marcações da seção 5 e a resposta esperada;
- `nsmo_casos.parquet`: os 718 casos da NSMO de 44% a 50% com score e reserva, com a
  finalidade, o menor score, a leitura da norma, a resposta esperada, a parte da
  avaliação e o grupo (vazio nos que ficam fora);
- `conjuntos_fixos.parquet`: os 44 casos da seção 9, pelo índice da base de origem.

Cada linha leva o índice original (`indice`), para o caso ser rastreado até a base
bruta.

In [62]:
PROCESSED.mkdir(parents=True, exist_ok=True)

colunas_hmda = COLUNAS_HMDA + [
    'grupo', 'decisao', 'faixa_dti', 'subfaixa_ate_43', 'cltv_90', 'cltv_publicado',
    'situacao_cltv', 'seguro_financiado', 'cltv_vazio', 'renda_vazia', 'aus_6',
    'onde_entra',
    'resposta_esperada', 'pendencias', 'motivos',
]
saida_hmda = casos[colunas_hmda].rename_axis('indice').reset_index()
saida_hmda.to_parquet(PROCESSED / 'hmda_casos.parquet', index=False)

colunas_nsmo = COLUNAS_NSMO + [
    'menor_score', 'loan_purpose', 'finalidade', 'tem_reserva', 'leitura',
    'situacao_cltv', 'onde_entra', 'resposta_esperada', 'pendencias', 'motivos',
    'grupo',
]
saida_nsmo = teste[colunas_nsmo].rename_axis('indice').reset_index()
saida_nsmo.to_parquet(PROCESSED / 'nsmo_casos.parquet', index=False)

conjuntos.to_parquet(PROCESSED / 'conjuntos_fixos.parquet', index=False)

for nome in ['hmda_casos.parquet', 'nsmo_casos.parquet', 'conjuntos_fixos.parquet']:
    caminho = PROCESSED / nome
    print(nome, f'{caminho.stat().st_size / 1e6:.2f} MB')
print('HMDA:', saida_hmda.shape, '| NSMO:', saida_nsmo.shape,
      '| conjuntos:', conjuntos.shape)

hmda_casos.parquet 13.49 MB
nsmo_casos.parquet 0.04 MB
conjuntos_fixos.parquet 0.00 MB
HMDA: (938903, 35) | NSMO: (718, 30) | conjuntos: (44, 4)


<a id="secao-11"></a>

## 11. Limitações

Ressalvas que não mudam o que o notebook faz, mas pesam na leitura do resultado. A
célula conta duas linhas da tabela que não saem de outra seção, com a decisão do
banco só no grupo de ajuste.

In [63]:
compras_150 = casos[(casos['loan_purpose'] == '1') & (casos['cltv'] > 150)]
print('compras com CLTV acima de 150% na base:', br(len(compras_150)))
no_ajuste_150 = compras_150[compras_150['grupo'] == 'ajuste']
print('  no ajuste, por decisão:', no_ajuste_150['decisao'].value_counts().to_dict())

ajuste = casos[casos['grupo'] == 'ajuste']
dti_baixo = ajuste[(ajuste['decisao'] == 'negado') & ajuste['motivo_dti_visivel']
                   & (ajuste['debt_to_income_ratio'] == '<20%')]
print('negados no ajuste que citam DTI, com DTI publicado abaixo de 20%:',
      br(len(dti_baixo)))

compras com CLTV acima de 150% na base: 75
  no ajuste, por decisão: {'negado': 12, 'aprovado': 7}


negados no ajuste que citam DTI, com DTI publicado abaixo de 20%: 109


| Limitação | Efeito | O que se faz |
|---|---|---|
| O HMDA não publica score, PTI nem reserva | até 43%, parte dos pedidos estouraria o PTI de 31% (cerca de 1 em 5 de 36% a 43%, pela NSMO) | resultado até 43% dividido em até 35% e 36% a 43% (`subfaixa_ate_43`) |
| O DTI vem em faixas abaixo de 36% e acima de 49% | a faixa de 50% a 60% mistura o 50%, permitido com dois fatores, com valores acima do teto manual | a faixa vai para o relatório à parte |
| O HMDA não diz se o CLTV inclui o seguro inicial; o jeito do banco é deduzido pelo pico | a classificação pela maioria é palpite nos 102 bancos com os dois picos e nos 90 sem pico, que contam como "sem o seguro" | coluna `cltv_publicado` e resultado da faixa separado por `seguro_financiado` |
| O sem saque tem limites que o HMDA não publica (dívida antiga somada aos custos, LTV de 85% para quem mora no imóvel há menos de 12 meses, II.A.8) | parte das negativas do sem saque dentro do teto não tem explicação no dado | nenhuma ação; ressalva |
| O HMDA não marca o Streamline | um Streamline com DTI publicado pode ter ficado na base | nenhuma ação; ressalva |
| 75 compras com CLTV acima de 150% (no ajuste, 7 aprovadas e 12 negadas) | o segundo financiamento de órgão público não tem teto de CLTV (Handbook p. 369 e 371), mas parte deve ser erro de preenchimento | seguem a regra geral, "segundo financiamento a confirmar" |
| 109 negados no ajuste que citam DTI com DTI publicado abaixo de 20% | o motivo declarado não bate com o número | ficam e contam como discordância |
| O HMDA não publica se o pedido foi aceito pelo sistema automático do FHA ou mandado ao analista (4.6) | o projeto compara a norma da análise manual com decisões que, em boa parte, vieram do sistema automático; um pedido acima de 60% aprovado pelo sistema conta como discordância, mesmo com o agente seguindo a norma, e pesa igual na norma em código e na regra fixa de DTI | a faixa de 50% a 60% já vai para o relatório à parte; o README e a Evaluation trazem a ressalva |
| Código 6 do `aus-1` (24.385 pedidos) | mistura análise manual com bancos que marcam 6 em tudo | sem subgrupo; campo fora do perfil |
| Renda vazia em 174 bancos do ajuste, um deles com 91 de 100 pedidos (4.6) | o "não informado" pesa mais nesse banco | fica com "renda: não informado" |
| Na NSMO, DTI e CLTV vêm em número inteiro | 98 cobre de 97,5 a 98,4, e a faixa do seguro na compra fica imprecisa | ressalva |
| Na NSMO, o score é VantageScore 3.0, e não o do banco | a leitura perto de 580 pode mudar | ressalva; a Evaluation pode medir também sem os casos perto de 580 |
| A NSMO só tem aprovados | o segundo teste não mede negativas, e 152 aprovações fora da tabela manual vieram de outro caminho | esses 152 ficam fora dos grupos |
| A reserva é o único fator compensatório que a NSMO mostra | os outros três fatores da norma não estão na base | fator não verificado vira "suspender pedindo a comprovação" |
| No sem saque da NSMO, o dado não confirma se o CLTV vem com o seguro (6.3) | os 14 casos com CLTV de 98 e 99 passariam a negar por garantia se a premissa estiver errada; parte deles já sai por estar fora da tabela manual | a Evaluation mostra a nota da NSMO também sem eles (`situacao_cltv` igual a "seguro financiado" no sem saque) |
| Os campos preenchidos e o formato do CLTV mostram de qual base vem o caso (8.1) | um prompt que cite as bases daria ao agente uma pista da resposta | o prompt não menciona as bases nem que a NSMO só tem aprovados (Modeling) |
| Os casos são de 2019 a 2025, e o gabarito usa a Update 18, de 2026 | só o teto do saque mudou no período (85% até 2019), e a NSMO só informa o ano do empréstimo; as versões entre novembro de 2020 e a Update 17 não foram conferidas | os 19 saques da NSMO acima do teto ficam fora do teste (6.1 e 6.3); ressalva |
| Escolhas do projeto conferidas no grupo de ajuste: corte de 2 vezes na renda (4.1), retirada das finalidades sem teto (4.5), margem do pico e faixa do seguro (4.6), saques da NSMO fora do teste (6.3) | o resultado no ajuste pode sair melhor que o da nota final | a Evaluation compara os dois grupos; uma diferença grande aponta para essas escolhas |

<a id="secao-12"></a>

## 12. Síntese

1. Dos 2.062.934 pedidos FHA de 2025 no HMDA, ficam 938.903, todos de residência principal em primeira hipoteca, com DTI
   publicado e finalidade com teto na tabela. Toda negativa que fica tem DTI ou
   garantia declarado e o campo que o confirma.
2. O grupo de ajuste tem 281.889 pedidos e o da nota final 657.014, sorteados antes de
   qualquer remoção.
3. O CLTV é lido pelo jeito do banco: 1.172 bancos publicam sem o seguro inicial e
   49 com ele. Nos primeiros, a faixa entre o teto e o teto com o seguro conta como
   acima do teto; a NSMO publica com o seguro.
4. São 360.096 pedidos medidos contra a decisão do banco (DTI até 43% ou acima de
   60%, e os refinanciamentos acima do teto do CLTV, que a norma nega direto),
   294.285 medidos contra a norma (fator compensatório ou segundo financiamento) e
   284.522 de 50% a 60%, no relatório à parte.
5. No grupo de ajuste, a norma em código acerta 94,8% na concordância com o banco,
   contra 93,5% da regra fixa de DTI e 81,8% de sempre aprovar. Com as negativas
   removidas de volta, os números ficam em 84,1%, 82,8% e 70,6%. O agente vai ser
   comparado com essas três respostas.
6. Na NSMO, o segundo teste tem 545 casos (90 de checagem e 455 contra a norma), 145
   no ajuste e 400 na nota final. Da checagem, 9 casos vão para o ajuste, para o
   prompt ver a resposta "aprovar com fator compensatório".
7. O agente lê um perfil de 12 campos, igual nas duas bases, sem a decisão, o motivo,
   o banco ou o nome da base. A linha do CLTV diz se ele vem com o seguro.
8. O próximo passo é o notebook 03, com o Handbook dividido em trechos, os metadados,
   os embeddings e o índice da norma.